In [2]:
import lime
import shap

In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -------- CONFIG --------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
    "models": ["LogReg", "DecisionTree", "KNN", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]
}

# ⚠️ Kendi bilgisayarındaki gerçek path
DATA_PATH = Path("~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv").expanduser()

def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(CONFIG["seed"])

BASE_OUT = Path(CONFIG["output_dir"])
SHARED = BASE_OUT / "_shared"

print("DATA_PATH:", DATA_PATH)
print("SHARED:", SHARED.resolve())

DATA_PATH: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\final_lexical_dns_whois_cleaned.csv
SHARED: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\_shared


In [2]:
# --- dataset yükle ---
df = pd.read_csv(DATA_PATH)
label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"]

assert label_col in df.columns, f"{label_col} yok"
assert url_col in df.columns, f"{url_col} yok"

feature_cols = [c for c in df.columns if c not in [url_col, label_col]]

X = df[feature_cols].copy()
y = df[label_col].astype(int).copy()

# --- split index dosyaları (Asama0 üretmişti) ---
train_idx_path = SHARED / "split_indices_train.csv"
test_idx_path  = SHARED / "split_indices_test.csv"

assert train_idx_path.exists(), f"Missing: {train_idx_path}"
assert test_idx_path.exists(),  f"Missing: {test_idx_path}"

train_idx = pd.read_csv(train_idx_path).iloc[:,0].astype(int).values
test_idx  = pd.read_csv(test_idx_path).iloc[:,0].astype(int).values

X_train_raw = X.iloc[train_idx].copy()
X_test_raw  = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

print("✅ Reload split OK")
print("X_train:", X_train_raw.shape, "X_test:", X_test_raw.shape, "n_features:", len(feature_cols))
print("y_test dist:", y_test.value_counts().to_dict())

✅ Reload split OK
X_train: (463936, 75) X_test: (115984, 75) n_features: 75
y_test dist: {0: 67815, 1: 48169}


In [3]:
# preprocessors
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# models
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=CONFIG["seed"], eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=CONFIG["seed"], n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=CONFIG["seed"], verbose=False
    )
}

LINEAR_DISTANCE = {"LogReg", "KNN"}  # scaler gerekenler

pipelines = {}
for name, clf in base_models.items():
    print("Fitting:", name)
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    pipe.fit(X_train_raw, y_train)
    pipelines[name] = pipe

print("\n✅ pipelines hazır:", list(pipelines.keys()))

Fitting: LogReg
Fitting: DecisionTree
Fitting: KNN
Fitting: RandomForest
Fitting: XGBoost
Fitting: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070447 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
Fitting: CatBoost

✅ pipelines hazır: ['LogReg', 'DecisionTree', 'KNN', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']


In [4]:
import os, json
import numpy as np
import pandas as pd
from pathlib import Path
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

# =========================
# AYARLAR
# =========================
LIME_TOTAL = 400
LIME_PER_GROUP = LIME_TOTAL // 2   # 200 correct + 200 wrong
RANDOM_SEED = 42

# LIME açıklamasında kaç feature kaydedilsin?
# SHAP ile karşılaştırmada genelde Top-20/Top-30 çok yeterli olur.
LIME_NUM_FEATURES = 30

# Eğitimden kaç satır LIME "training_data" olarak kullanılacak?
# Çok büyük tutma; discretization + sampling maliyeti artar.
LIME_TRAIN_SAMPLE = 5000

# Çok fazla PNG üretmeyelim. İstersen 10/10 gibi az sayıda görsel al.
MAKE_PNG = True
PNG_PER_GROUP = 10  # correct/wrong başına üretilecek görsel sayısı

# Çıktı kök dizini (Asama0'ındaki BASE_OUT'a uyarlayabilirsin)
# Örn: BASE_OUT = Path("outputs")
# Eğer sende zaten BASE_OUT varsa bunu tekrar tanımlama.
BASE_OUT = Path("outputs")

rng = np.random.RandomState(RANDOM_SEED)

def pick_balanced_indices(pred, y_true, per_group, rng):
    """200 correct + 200 wrong gibi dengeli örnek seçimi."""
    correct_idx = np.where(pred == y_true)[0]
    wrong_idx   = np.where(pred != y_true)[0]

    n_c = min(per_group, len(correct_idx))
    n_w = min(per_group, len(wrong_idx))

    pick_c = rng.choice(correct_idx, size=n_c, replace=False) if n_c > 0 else np.array([], dtype=int)
    pick_w = rng.choice(wrong_idx,   size=n_w, replace=False) if n_w > 0 else np.array([], dtype=int)

    return pick_c, pick_w, len(correct_idx), len(wrong_idx)

def build_lime_explainer(X_train_df, feature_cols):
    """LIME explainer'ı train data üzerinden kur."""
    n_train = len(X_train_df)
    n_use = min(LIME_TRAIN_SAMPLE, n_train)
    train_sample = X_train_df.sample(n=n_use, random_state=RANDOM_SEED)

    explainer = LimeTabularExplainer(
        training_data=train_sample.values,
        feature_names=feature_cols,
        class_names=["benign", "phishing"],
        discretize_continuous=True,
        mode="classification"
    )
    return explainer, n_use

def predict_fn_factory(pipe, feature_cols):
    """LIME predict_fn: numpy -> DataFrame -> pipe.predict_proba"""
    def predict_fn(z):
        z_df = pd.DataFrame(z, columns=feature_cols)
        return pipe.predict_proba(z_df)
    return predict_fn

def run_lime_for_model(model_name, pipe, X_train_raw, X_test_raw, y_test, feature_cols):
    """
    400 örnek (200 correct + 200 wrong) için LIME çıkarır.
    - Detaylı uzun format kaydeder (sample_id, feature_name, weight...)
    - İsteğe bağlı az sayıda PNG üretir.
    """
    out_dir = BASE_OUT / model_name / "lime_local_v2_400"
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "png_correct").mkdir(parents=True, exist_ok=True)
    (out_dir / "png_wrong").mkdir(parents=True, exist_ok=True)

    # Predict
    proba = pipe.predict_proba(X_test_raw)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    pick_c, pick_w, n_correct, n_wrong = pick_balanced_indices(pred, y_test, LIME_PER_GROUP, rng)

    # LIME explainer
    explainer, n_train_used = build_lime_explainer(X_train_raw[feature_cols], feature_cols)
    predict_fn = predict_fn_factory(pipe, feature_cols)

    # Samples dataframe
    rows = []
    for i in pick_c:
        rows.append(("correct", int(i)))
    for i in pick_w:
        rows.append(("wrong", int(i)))

    df_samples = pd.DataFrame(rows, columns=["group", "sample_id"])
    df_samples["y_true"] = y_test[df_samples["sample_id"].values]
    df_samples["y_pred"] = pred[df_samples["sample_id"].values]
    df_samples["p_phishing"] = proba[df_samples["sample_id"].values]

    # LIME explanations (long format)
    long_records = []

    # PNG için sayaç
    png_count = {"correct": 0, "wrong": 0}

    for r in df_samples.itertuples(index=False):
        group = r.group
        i = r.sample_id
        y_true = int(r.y_true)
        y_pred = int(r.y_pred)
        p1 = float(r.p_phishing)

        x = X_test_raw.iloc[i][feature_cols].values

        # top_labels=2 -> iki sınıfı da hesaplatır, bazı edge-case hatalarını azaltır
        exp = explainer.explain_instance(
            data_row=x,
            predict_fn=predict_fn,
            num_features=LIME_NUM_FEATURES,
            top_labels=2
        )

        # Hangi label'ı kaydedelim?
        # SHAP ile karşılaştırmada genelde "phishing sınıfı" (label=1) daha anlamlı.
        # Ama exp o labelı üretmemiş olabilir; o yüzden fallback yapıyoruz.
        target_label = 1 if 1 in exp.available_labels() else exp.available_labels()[0]

        # as_map -> (feature_index, weight) verir (string bin yerine gerçek feature)
        fmap = exp.as_map()[target_label]   # list of tuples: (feature_idx, weight)
        # Mutlak ağırlığa göre sırala (rank için)
        fmap_sorted = sorted(fmap, key=lambda t: abs(t[1]), reverse=True)

        for rank, (f_idx, w) in enumerate(fmap_sorted, start=1):
            long_records.append({
                "model": model_name,
                "group": group,
                "sample_id": i,
                "y_true": y_true,
                "y_pred": y_pred,
                "p_phishing": p1,
                "explained_label": int(target_label),
                "feature_idx": int(f_idx),
                "feature_name": feature_cols[int(f_idx)],
                "weight": float(w),
                "abs_weight": float(abs(w)),
                "rank": int(rank)
            })

        # Az sayıda PNG üret
        if MAKE_PNG and png_count[group] < PNG_PER_GROUP:
            fig = exp.as_pyplot_figure(label=target_label)
            plt.title(f"{model_name} LIME | {group} | row={i} | y={y_true} pred={y_pred} p={p1:.3f}")
            plt.tight_layout()
            fname = f"{group}_{i}_y{y_true}_p{p1:.3f}.png"
            if group == "correct":
                plt.savefig(out_dir / "png_correct" / fname, dpi=220)
            else:
                plt.savefig(out_dir / "png_wrong" / fname, dpi=220)
            plt.close()
            png_count[group] += 1

    # Save
    df_long = pd.DataFrame(long_records)
    df_long.to_csv(out_dir / "lime_explanations_long.csv", index=False, encoding="utf-8")
    df_samples.to_csv(out_dir / "lime_samples.csv", index=False, encoding="utf-8")

    # Parquet (opsiyonel ama çok faydalı: hızlı okuma/karşılaştırma)
    try:
        df_long.to_parquet(out_dir / "lime_explanations_long.parquet", index=False)
    except Exception as e:
        print("⚠️ Parquet yazılamadı (opsiyonel):", e)

    meta = {
        "model": model_name,
        "lime_total_target": LIME_TOTAL,
        "lime_per_group_target": LIME_PER_GROUP,
        "lime_num_features_saved": LIME_NUM_FEATURES,
        "lime_train_sample_used": int(n_train_used),
        "test_correct_total": int(n_correct),
        "test_wrong_total": int(n_wrong),
        "picked_correct": int((df_samples["group"]=="correct").sum()),
        "picked_wrong": int((df_samples["group"]=="wrong").sum()),
        "random_seed": int(RANDOM_SEED),
        "make_png": bool(MAKE_PNG),
        "png_per_group": int(PNG_PER_GROUP),
    }
    with open(out_dir / "lime_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"✅ {model_name} LIME done -> {out_dir}")
    print("   saved:", "lime_explanations_long.csv, lime_samples.csv, lime_meta.json")
    return out_dir

# =========================
# SENİN MODEL LOOP'UNA BAĞLANTI
# =========================
# Aşağıdaki iki değişkenin sende zaten var olması gerekiyor:
# 1) pipelines: {"LogReg": pipe, "RandomForest": pipe, ...} gibi
# 2) df_train / df_test veya X_train_raw, X_test_raw, y_test
#
# ÖNEMLİ: X_train_raw ve X_test_raw DataFrame olmalı ve feature_cols kolonlarını içermeli.

# Örnek beklenen değişkenler:
# pipelines = {"LightGBM": lgbm_pipe, "CatBoost": cat_pipe, ...}
# X_train_raw = df_train[feature_cols]
# X_test_raw  = df_test[feature_cols]
# y_test      = y_test_array_or_series

SKIP_MODELS = {"KNN"}

for model_name, pipe in pipelines.items():
    if model_name in SKIP_MODELS:
        print(f"⏭️ Skipped: {model_name}")
        continue

    run_lime_for_model(
        model_name=model_name,
        pipe=pipe,
        X_train_raw=X_train_raw,
        X_test_raw=X_test_raw,
        y_test=np.array(y_test),
        feature_cols=list(feature_cols)
    )

⚠️ Parquet yazılamadı (opsiyonel): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
✅ LogReg LIME done -> outputs\LogReg\lime_local_v2_400
   saved: lime_explanations_long.csv, lime_samples.csv, lime_meta.json
⚠️ Parquet yazılamadı (opsiyonel): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependen

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

⚠️ Parquet yazılamadı (opsiyonel): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
✅ LightGBM LIME done -> outputs\LightGBM\lime_local_v2_400
   saved: lime_explanations_long.csv, lime_samples.csv, lime_meta.json
⚠️ Parquet yazılamadı (opsiyonel): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional depe

In [6]:
import os, json
import numpy as np
import pandas as pd
from pathlib import Path
import shap

BASE_OUT = Path("outputs")
TOPK = 30
BG_SIZE = 2000
SEED = 42

def coerce_shap_2d(sv, n_features):
    """
    SHAP çıktısını (n_samples, n_features) 2D matrise indirger.
    Olası formatlar:
      - list[len=2] of (n_samples, n_features): -> sv[1]
      - np.array 2D: (n_samples, n_features)
      - np.array 3D: (n_samples, n_features, 2) veya (n_samples, 2, n_features)
    """
    # 1) list -> binary class: pick class 1
    if isinstance(sv, list):
        if len(sv) == 2:
            sv = sv[1]
        else:
            # multi-class ise 1 yoksa 0 al
            sv = sv[min(1, len(sv)-1)]

    sv = np.asarray(sv)

    # 2) already 2D
    if sv.ndim == 2:
        if sv.shape[1] != n_features:
            raise ValueError(f"SHAP 2D shape mismatch: {sv.shape} vs n_features={n_features}")
        return sv

    # 3) 3D cases
    if sv.ndim == 3:
        # Case A: (n_samples, n_features, n_classes)
        if sv.shape[1] == n_features:
            # pick class 1 if exists else 0
            cls_axis = 2
            cls = 1 if sv.shape[cls_axis] > 1 else 0
            out = sv[:, :, cls]
            return out

        # Case B: (n_samples, n_classes, n_features)
        if sv.shape[2] == n_features:
            cls_axis = 1
            cls = 1 if sv.shape[cls_axis] > 1 else 0
            out = sv[:, cls, :]
            return out

        raise ValueError(f"Unsupported SHAP 3D shape: {sv.shape} with n_features={n_features}")

    raise ValueError(f"Unsupported SHAP ndim={sv.ndim}, shape={sv.shape}")

def _get_model_and_data(pipe, X_train_raw, X_test_raw, sample_ids):
    pre = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]

    X_bg = X_train_raw.sample(n=min(BG_SIZE, len(X_train_raw)), random_state=SEED)
    X_bg_t = pre.transform(X_bg)

    X_s = X_test_raw.iloc[sample_ids]
    X_s_t = pre.transform(X_s)
    return model, X_bg_t, X_s_t

def build_shap_local_long_for_model(model_name, pipe, X_train_raw, X_test_raw, y_test, feature_cols):
    lime_dir = BASE_OUT / model_name / "lime_local_v2_400"
    samples_path = lime_dir / "lime_samples.csv"
    if not samples_path.exists():
        print(f"⏭️ {model_name}: lime_samples.csv yok, atlandı.")
        return None

    df_samples = pd.read_csv(samples_path)
    sample_ids = df_samples["sample_id"].astype(int).tolist()

    out_dir = BASE_OUT / model_name / "shap_local_v2_400"
    out_dir.mkdir(parents=True, exist_ok=True)

    model, X_bg_t, X_s_t = _get_model_and_data(pipe, X_train_raw, X_test_raw, sample_ids)
    n_features = len(feature_cols)

    # Explainer seçimi
    if model_name in {"DecisionTree", "RandomForest", "LightGBM", "XGBoost", "CatBoost"}:
        explainer = shap.TreeExplainer(model, data=X_bg_t)
        explainer_type = "TreeExplainer"
        sv_raw = explainer.shap_values(X_s_t)
    else:
        explainer = shap.LinearExplainer(model, X_bg_t)
        explainer_type = "LinearExplainer"
        sv_raw = explainer.shap_values(X_s_t)

    # Robust 2D'ye indir
    shap_vals = coerce_shap_2d(sv_raw, n_features=n_features)

    # long format
    long_records = []
    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    for row_i, sid in enumerate(sample_ids):
        vals = shap_vals[row_i]
        abs_vals = np.abs(vals)
        top_idx = np.argsort(abs_vals)[::-1][:TOPK]

        for rank, f_idx in enumerate(top_idx, start=1):
            fi = int(f_idx)
            long_records.append({
                "model": model_name,
                "sample_id": int(sid),
                "y_true": int(y_arr[sid]),
                "feature_name": feature_cols[fi],
                "shap_value": float(vals[fi]),
                "abs_shap": float(abs_vals[fi]),
                "rank": int(rank),
                "explainer": explainer_type,
            })

    df_long = pd.DataFrame(long_records)
    df_long.to_csv(out_dir / "shap_local_long.csv", index=False, encoding="utf-8")

    meta = {
        "model": model_name,
        "explainer": explainer_type,
        "topk_saved": TOPK,
        "bg_size": int(min(BG_SIZE, len(X_train_raw))),
        "n_samples": int(len(sample_ids)),
        "shap_raw_type": str(type(sv_raw)),
        "shap_raw_shape": getattr(np.asarray(sv_raw), "shape", None) if not isinstance(sv_raw, list) else [np.asarray(x).shape for x in sv_raw],
    }
    with open(out_dir / "shap_local_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"✅ {model_name} SHAP local saved -> {out_dir}")
    return out_dir

# ---- ÇALIŞTIR ----
for model_name, pipe in pipelines.items():
    build_shap_local_long_for_model(
        model_name=model_name,
        pipe=pipe,
        X_train_raw=X_train_raw,
        X_test_raw=X_test_raw,
        y_test=y_test,
        feature_cols=list(feature_cols),
    )

✅ LogReg SHAP local saved -> outputs\LogReg\shap_local_v2_400
✅ DecisionTree SHAP local saved -> outputs\DecisionTree\shap_local_v2_400
⏭️ KNN: lime_samples.csv yok, atlandı.


100%|===================| 798/800 [11:08<00:01]        

✅ RandomForest SHAP local saved -> outputs\RandomForest\shap_local_v2_400


ValueError: could not convert string to float: '[4.1530943E-1]'

In [7]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import shap

BASE_OUT = Path("outputs")
TOPK = 30
BG_SIZE = 2000
SEED = 42

def coerce_shap_2d(sv, n_features):
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]
    sv = np.asarray(sv)

    if sv.ndim == 2:
        if sv.shape[1] != n_features:
            raise ValueError(f"SHAP 2D shape mismatch: {sv.shape} vs n_features={n_features}")
        return sv

    if sv.ndim == 3:
        # (n, f, c)
        if sv.shape[1] == n_features:
            cls = 1 if sv.shape[2] > 1 else 0
            return sv[:, :, cls]
        # (n, c, f)
        if sv.shape[2] == n_features:
            cls = 1 if sv.shape[1] > 1 else 0
            return sv[:, cls, :]
        raise ValueError(f"Unsupported SHAP 3D shape: {sv.shape} with n_features={n_features}")

    raise ValueError(f"Unsupported SHAP ndim={sv.ndim}, shape={sv.shape}")

def get_bg_transformed(pipe, X_train_raw):
    pre = pipe.named_steps["preprocess"]
    X_bg = X_train_raw.sample(n=min(BG_SIZE, len(X_train_raw)), random_state=SEED)
    X_bg_t = pre.transform(X_bg)
    return X_bg, X_bg_t

def get_samples_from_lime(model_name):
    lime_dir = BASE_OUT / model_name / "lime_local_v2_400"
    samples_path = lime_dir / "lime_samples.csv"
    if not samples_path.exists():
        print(f"⏭️ {model_name}: lime_samples.csv yok, atlandı.")
        return None
    return pd.read_csv(samples_path)

def save_meta(out_dir, meta):
    with open(out_dir / "shap_local_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

def run_tree_or_linear(model_name, pipe, X_train_raw, X_test_raw, y_test, feature_cols, df_samples, out_dir):
    pre = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]
    sample_ids = df_samples["sample_id"].astype(int).tolist()

    # background + samples (transformed)
    _, X_bg_t = get_bg_transformed(pipe, X_train_raw)
    X_s_t = pre.transform(X_test_raw.iloc[sample_ids])

    n_features = len(feature_cols)
    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    # explainer
    if model_name in {"DecisionTree", "RandomForest", "LightGBM", "CatBoost"}:
        explainer = shap.TreeExplainer(model, data=X_bg_t)
        explainer_type = "TreeExplainer"
        sv_raw = explainer.shap_values(X_s_t)
    else:
        explainer = shap.LinearExplainer(model, X_bg_t)
        explainer_type = "LinearExplainer"
        sv_raw = explainer.shap_values(X_s_t)

    shap_vals = coerce_shap_2d(sv_raw, n_features=n_features)

    # long
    long_records = []
    for row_i, sid in enumerate(sample_ids):
        vals = shap_vals[row_i]
        abs_vals = np.abs(vals)
        top_idx = np.argsort(abs_vals)[::-1][:TOPK]
        for rank, f_idx in enumerate(top_idx, start=1):
            fi = int(f_idx)
            long_records.append({
                "model": model_name,
                "sample_id": int(sid),
                "y_true": int(y_arr[sid]),
                "feature_name": feature_cols[fi],
                "shap_value": float(vals[fi]),
                "abs_shap": float(abs_vals[fi]),
                "rank": int(rank),
                "explainer": explainer_type,
            })

    df_long = pd.DataFrame(long_records)
    df_long.to_csv(out_dir / "shap_local_long.csv", index=False, encoding="utf-8")

    meta = {
        "model": model_name,
        "explainer": explainer_type,
        "topk_saved": TOPK,
        "bg_size": int(min(BG_SIZE, len(X_train_raw))),
        "n_samples": int(len(sample_ids)),
        "note": "TreeExplainer/LinearExplainer path",
        "shap_raw_shape": [np.asarray(x).shape for x in sv_raw] if isinstance(sv_raw, list) else np.asarray(sv_raw).shape,
    }
    save_meta(out_dir, meta)
    print(f"✅ {model_name} SHAP local saved -> {out_dir}")

def run_xgb_fallback(model_name, pipe, X_train_raw, X_test_raw, y_test, feature_cols, df_samples, out_dir):
    """
    XGBoost için TreeExplainer base_score hatasını bypass eder:
    - shap.Explainer(predict_fn, bg_transformed) -> genelde PermutationExplainer seçer
    - 400 örnek için makul süre
    """
    pre = pipe.named_steps["preprocess"]
    model = pipe.named_steps["model"]
    sample_ids = df_samples["sample_id"].astype(int).tolist()

    # background (transformed) + sample (transformed)
    _, X_bg_t = get_bg_transformed(pipe, X_train_raw)
    X_s_t = pre.transform(X_test_raw.iloc[sample_ids])

    # predict_fn over transformed space (model expects transformed)
    def predict_fn(z):
        # z: numpy matrix in transformed feature space
        # return proba for both classes
        return model.predict_proba(z)

    explainer = shap.Explainer(predict_fn, X_bg_t)  # PermutationExplainer seçebilir
    explainer_type = type(explainer).__name__

    exp = explainer(X_s_t)  # shap.Explanation
    # exp.values şekli: (n, f, c) veya (n, f)
    sv_raw = exp.values
    shap_vals = coerce_shap_2d(sv_raw, n_features=len(feature_cols))

    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    long_records = []
    for row_i, sid in enumerate(sample_ids):
        vals = shap_vals[row_i]
        abs_vals = np.abs(vals)
        top_idx = np.argsort(abs_vals)[::-1][:TOPK]
        for rank, f_idx in enumerate(top_idx, start=1):
            fi = int(f_idx)
            long_records.append({
                "model": model_name,
                "sample_id": int(sid),
                "y_true": int(y_arr[sid]),
                "feature_name": feature_cols[fi],
                "shap_value": float(vals[fi]),
                "abs_shap": float(abs_vals[fi]),
                "rank": int(rank),
                "explainer": explainer_type,
            })

    df_long = pd.DataFrame(long_records)
    df_long.to_csv(out_dir / "shap_local_long.csv", index=False, encoding="utf-8")

    meta = {
        "model": model_name,
        "explainer": explainer_type,
        "topk_saved": TOPK,
        "bg_size": int(min(BG_SIZE, len(X_train_raw))),
        "n_samples": int(len(sample_ids)),
        "note": "XGBoost fallback via shap.Explainer(predict_fn, bg)",
        "sv_raw_shape": np.asarray(sv_raw).shape,
    }
    save_meta(out_dir, meta)
    print(f"✅ {model_name} SHAP local saved (fallback) -> {out_dir}")

# -----------------------------
# MAIN: sadece eksikleri çalıştır
# -----------------------------
for model_name, pipe in pipelines.items():
    # KNN zaten yok; varsa da lime_samples olmadığı için skip olur.
    df_samples = get_samples_from_lime(model_name)
    if df_samples is None:
        continue

    out_dir = BASE_OUT / model_name / "shap_local_v2_400"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_dir / "shap_local_long.csv"
    if out_csv.exists():
        print(f"⏭️ {model_name}: zaten var -> {out_csv}")
        continue

    try:
        if model_name == "XGBoost":
            run_xgb_fallback(model_name, pipe, X_train_raw, X_test_raw, y_test, list(feature_cols), df_samples, out_dir)
        else:
            run_tree_or_linear(model_name, pipe, X_train_raw, X_test_raw, y_test, list(feature_cols), df_samples, out_dir)
    except Exception as e:
        # CatBoost gibi bazı durumlarda TreeExplainer patlarsa fallback dene
        if model_name in {"CatBoost", "LightGBM"}:
            print(f"⚠️ {model_name}: Tree/Linear path hata verdi, fallback deneniyor. Hata: {e}")
            try:
                run_xgb_fallback(model_name, pipe, X_train_raw, X_test_raw, y_test, list(feature_cols), df_samples, out_dir)
            except Exception as e2:
                print(f"❌ {model_name}: fallback da başarısız: {e2}")
        else:
            print(f"❌ {model_name}: başarısız: {e}")

⏭️ LogReg: zaten var -> outputs\LogReg\shap_local_v2_400\shap_local_long.csv
⏭️ DecisionTree: zaten var -> outputs\DecisionTree\shap_local_v2_400\shap_local_long.csv
⏭️ KNN: lime_samples.csv yok, atlandı.
⏭️ RandomForest: zaten var -> outputs\RandomForest\shap_local_v2_400\shap_local_long.csv


PermutationExplainer explainer: 401it [01:02,  6.37it/s]                                                               


✅ XGBoost SHAP local saved (fallback) -> outputs\XGBoost\shap_local_v2_400


100%|===================| 399/400 [00:15<00:00]       C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


⚠️ LightGBM: Tree/Linear path hata verdi, fallback deneniyor. Hata: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. This check failed because for one of the samples the sum of the SHAP values was -4.074929, while the model output was -4.289303. If this difference is acceptable you can set check_additivity=False to disable this check.


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

✅ LightGBM SHAP local saved (fallback) -> outputs\LightGBM\shap_local_v2_400


 98%|===================| 392/400 [00:37<00:00]        

✅ CatBoost SHAP local saved -> outputs\CatBoost\shap_local_v2_400


In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

BASE_OUT = Path("outputs")
COMPARE_DIR = BASE_OUT / "_shared" / "lime_shap_compare_400"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

TOPK = 10  # karşılaştırmayı Top-10 üzerinden raporlamak genelde daha temiz

def load_lime_long(model_name):
    p = BASE_OUT / model_name / "lime_local_v2_400" / "lime_explanations_long.csv"
    return pd.read_csv(p)

def load_lime_samples(model_name):
    p = BASE_OUT / model_name / "lime_local_v2_400" / "lime_samples.csv"
    return pd.read_csv(p)

def load_shap_long(model_name):
    p = BASE_OUT / model_name / "shap_local_v2_400" / "shap_local_long.csv"
    return pd.read_csv(p)

def topk_set(df, sid, value_col, k):
    sub = df[df["sample_id"] == sid].sort_values(value_col, ascending=False).head(k)
    return set(sub["feature_name"].tolist()), sub.set_index("feature_name")

all_rows = []

for model_name in pipelines.keys():
    lime_path = BASE_OUT / model_name / "lime_local_v2_400" / "lime_explanations_long.csv"
    shap_path = BASE_OUT / model_name / "shap_local_v2_400" / "shap_local_long.csv"
    if not lime_path.exists() or not shap_path.exists():
        print(f"⏭️ {model_name}: lime/shap dosyası eksik, atlandı.")
        continue

    df_lime = load_lime_long(model_name)
    df_shap = load_shap_long(model_name)
    df_samp = load_lime_samples(model_name)[["sample_id", "group", "y_true", "y_pred", "p_phishing"]]

    # LIME'de explained_label=1 filtrele (biz öyle kaydetmiştik)
    if "explained_label" in df_lime.columns:
        df_lime = df_lime[df_lime["explained_label"] == 1].copy()

    sample_ids = df_samp["sample_id"].astype(int).tolist()

    for sid in sample_ids:
        lime_top_set, lime_map = topk_set(df_lime, sid, "abs_weight", TOPK)
        shap_top_set, shap_map = topk_set(df_shap, sid, "abs_shap", TOPK)

        inter = lime_top_set.intersection(shap_top_set)
        overlap = len(inter) / TOPK

        # Spearman: union üzerinde abs değer vektörü
        union = sorted(lime_top_set.union(shap_top_set))
        lime_vec = pd.Series({f: abs(lime_map.loc[f, "weight"]) if f in lime_map.index else 0.0 for f in union})
        shap_vec = pd.Series({f: abs(shap_map.loc[f, "shap_value"]) if f in shap_map.index else 0.0 for f in union})

        # Spearman (pandas)
        rho = lime_vec.rank().corr(shap_vec.rank(), method="pearson")
        if pd.isna(rho):
            rho = 0.0

        # Sign agreement: intersection'da işaret uyumu
        if len(inter) > 0:
            agree = []
            for f in inter:
                s1 = np.sign(float(lime_map.loc[f, "weight"]))
                s2 = np.sign(float(shap_map.loc[f, "shap_value"]))
                agree.append(1 if s1 == s2 else 0)
            sign_agree = float(np.mean(agree))
        else:
            sign_agree = 0.0

        g = df_samp[df_samp["sample_id"] == sid].iloc[0]
        all_rows.append({
            "model": model_name,
            "sample_id": int(sid),
            "group": g["group"],
            "y_true": int(g["y_true"]),
            "y_pred": int(g["y_pred"]),
            "p_phishing": float(g["p_phishing"]),
            "topk": TOPK,
            "overlap_ratio": overlap,
            "spearman_rho": float(rho),
            "sign_agreement": sign_agree
        })

df_cmp = pd.DataFrame(all_rows)
df_cmp.to_csv(COMPARE_DIR / "lime_shap_compare_per_sample.csv", index=False, encoding="utf-8")

# ---- Özet ----
df_summary = (
    df_cmp.groupby(["model", "group"])
    .agg(
        n=("sample_id", "count"),
        overlap_mean=("overlap_ratio", "mean"),
        overlap_std=("overlap_ratio", "std"),
        rho_mean=("spearman_rho", "mean"),
        rho_std=("spearman_rho", "std"),
        sign_mean=("sign_agreement", "mean"),
        sign_std=("sign_agreement", "std"),
    )
    .reset_index()
)
df_summary.to_csv(COMPARE_DIR / "lime_shap_compare_summary.csv", index=False, encoding="utf-8")

print("✅ Saved:", (COMPARE_DIR / "lime_shap_compare_per_sample.csv"))
print("✅ Saved:", (COMPARE_DIR / "lime_shap_compare_summary.csv"))

# ---- Grafik 1: Model bazlı overlap ortalaması (correct+wrong ayrı) ----
for metric, fname, title in [
    ("overlap_ratio", "plot_overlap.png", "Top-K Overlap (LIME vs SHAP)"),
    ("spearman_rho", "plot_spearman.png", "Spearman Rank Correlation (abs values)"),
    ("sign_agreement", "plot_sign_agreement.png", "Sign Agreement on Intersection"),
]:
    pivot = df_cmp.groupby(["model", "group"])[metric].mean().unstack("group")
    ax = pivot.plot(kind="bar", rot=45, figsize=(10, 4))
    ax.set_title(title)
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.savefig(COMPARE_DIR / fname, dpi=220)
    plt.close()

print("✅ Plots saved in:", COMPARE_DIR)

⏭️ KNN: lime/shap dosyası eksik, atlandı.
✅ Saved: outputs\_shared\lime_shap_compare_400\lime_shap_compare_per_sample.csv
✅ Saved: outputs\_shared\lime_shap_compare_400\lime_shap_compare_summary.csv
✅ Plots saved in: outputs\_shared\lime_shap_compare_400


In [9]:
import pandas as pd
from pathlib import Path

p = Path("outputs/_shared/lime_shap_compare_400/lime_shap_compare_summary.csv")
df = pd.read_csv(p)

def fmt(m, s):
    return f"{m:.3f} ± {s:.3f}"

df_out = df.copy()
df_out["Overlap (mean±std)"] = df_out.apply(lambda r: fmt(r["overlap_mean"], r["overlap_std"]), axis=1)
df_out["Spearman ρ (mean±std)"] = df_out.apply(lambda r: fmt(r["rho_mean"], r["rho_std"]), axis=1)
df_out["Sign Agree (mean±std)"] = df_out.apply(lambda r: fmt(r["sign_mean"], r["sign_std"]), axis=1)

df_out = df_out[["model","group","n","Overlap (mean±std)","Spearman ρ (mean±std)","Sign Agree (mean±std)"]]
df_out = df_out.sort_values(["model","group"])

df_out.to_csv("outputs/_shared/lime_shap_compare_400/table_for_paper.csv", index=False, encoding="utf-8")
df_out

,model,group,n,Overlap (mean±std),Spearman ρ (mean±std),Sign Agree (mean±std)
0,CatBoost,correct,200,0.368 ± 0.092,-0.179 ± 0.182,0.997 ± 0.027
1,CatBoost,wrong,200,0.365 ± 0.104,-0.237 ± 0.189,0.998 ± 0.026
2,DecisionTree,correct,200,0.279 ± 0.108,-0.496 ± 0.188,0.977 ± 0.123
3,DecisionTree,wrong,200,0.252 ± 0.092,-0.588 ± 0.158,0.910 ± 0.188
4,LightGBM,correct,200,0.304 ± 0.112,-0.346 ± 0.205,0.995 ± 0.071
5,LightGBM,wrong,200,0.282 ± 0.088,-0.399 ± 0.169,0.997 ± 0.033
6,LogReg,correct,200,0.310 ± 0.110,-0.378 ± 0.189,1.000 ± 0.000
7,LogReg,wrong,200,0.318 ± 0.101,-0.384 ± 0.220,1.000 ± 0.000
8,RandomForest,correct,200,0.461 ± 0.128,0.019 ± 0.286,0.999 ± 0.010
9,RandomForest,wrong,200,0.436 ± 0.113,-0.061 ± 0.257,0.998 ± 0.020


In [16]:
import os, json, time
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.RandomState(SEED)

BASE_OUT = Path("outputs")
ENS_ROOT = BASE_OUT / "ENSEMBLES"
ENS_ROOT.mkdir(parents=True, exist_ok=True)

# Performans / süre kontrolü için (global SHAP sample)
GLOBAL_SHAP_TEST_SAMPLE = 3000   # istersen 5000 yap, ama daha yavaş olur
BG_SIZE = 2000                  # background sample
TOPK_GLOBAL = 20
TOPK_LOCAL = 30
TOPK_COMPARE = 10

LIME_TOTAL = 400
LIME_PER_GROUP = 200
LIME_NUM_FEATURES = 30
LIME_NUM_SAMPLES = 2000
LIME_TRAIN_SAMPLE = 5000

MAKE_PNG = True   # global shap grafikleri için True öneririm

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_csv(path: Path, df: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8")

def exists(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0

print("✅ Cache root:", ENS_ROOT.resolve())

✅ Cache root: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\ENSEMBLES


In [17]:
def ensemble_soft_predict_proba(X_df):
    # pipelines: senin eğitimli modeller sözlüğün (KNN yok)
    probs = []
    for name, pipe in pipelines.items():
        p = pipe.predict_proba(X_df)[:, 1]
        probs.append(p)
    probs = np.vstack(probs)
    mean_prob = probs.mean(axis=0)
    return np.column_stack([1 - mean_prob, mean_prob])

In [18]:
from sklearn.metrics import roc_auc_score

def get_or_make_samples_400(ens_dir: Path, predict_proba_fn):
    samples_path = ens_dir / "samples_400.csv"
    if exists(samples_path):
        print("⏭️ samples_400 cache var:", samples_path)
        return pd.read_csv(samples_path)

    p = predict_proba_fn(X_test_raw)[:, 1]
    pred = (p >= 0.5).astype(int)
    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    correct_idx = np.where(pred == y_arr)[0]
    wrong_idx   = np.where(pred != y_arr)[0]

    n_c = min(LIME_PER_GROUP, len(correct_idx))
    n_w = min(LIME_PER_GROUP, len(wrong_idx))

    pick_c = rng.choice(correct_idx, size=n_c, replace=False)
    pick_w = rng.choice(wrong_idx,   size=n_w, replace=False)

    rows = [("correct", int(i)) for i in pick_c] + [("wrong", int(i)) for i in pick_w]
    df_samples = pd.DataFrame(rows, columns=["group", "sample_id"])
    df_samples["y_true"] = y_arr[df_samples["sample_id"].values]
    df_samples["y_pred"] = pred[df_samples["sample_id"].values]
    df_samples["p_phishing"] = p[df_samples["sample_id"].values]

    save_csv(samples_path, df_samples)
    print("✅ samples_400 saved:", samples_path)
    return df_samples

def get_or_make_bg_idx(ens_dir: Path):
    bg_path = ens_dir / "bg_idx.csv"
    if exists(bg_path):
        return pd.read_csv(bg_path)["idx"].astype(int).tolist()

    n_bg = min(BG_SIZE, len(X_train_raw))
    idx = rng.choice(np.arange(len(X_train_raw)), size=n_bg, replace=False)
    df = pd.DataFrame({"idx": idx.astype(int)})
    save_csv(bg_path, df)
    return df["idx"].tolist()

def ensure_ensemble_dir(name: str, config: dict):
    ens_dir = ENS_ROOT / name
    ens_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = ens_dir / "config.json"
    if not exists(cfg_path):
        save_json(cfg_path, config)
    return ens_dir

In [19]:
from lime.lime_tabular import LimeTabularExplainer

def run_lime_cached(ens_name: str, ens_dir: Path, predict_proba_fn, df_samples: pd.DataFrame):
    out_dir = ens_dir / "lime"
    out_long = out_dir / "lime_explanations_long.csv"
    out_samples = out_dir / "lime_samples.csv"
    out_meta = out_dir / "lime_meta.json"

    if exists(out_long) and exists(out_samples):
        print(f"⏭️ {ens_name} LIME cache var.")
        return

    feature_cols = list(feature_cols_global)  # aşağıda tanımlayacağız
    # train sample
    train_sample = X_train_raw[feature_cols].sample(
        n=min(LIME_TRAIN_SAMPLE, len(X_train_raw)),
        random_state=SEED
    )

    explainer = LimeTabularExplainer(
        training_data=train_sample.values,
        feature_names=feature_cols,
        class_names=["benign", "phishing"],
        discretize_continuous=True,
        mode="classification"
    )

    sample_ids = df_samples["sample_id"].astype(int).tolist()

    long_records = []
    for sid in sample_ids:
        x = X_test_raw.iloc[sid][feature_cols].values

        exp = explainer.explain_instance(
            data_row=x,
            predict_fn=lambda z: predict_proba_fn(pd.DataFrame(z, columns=feature_cols)),
            num_features=LIME_NUM_FEATURES,
            top_labels=2,
            num_samples=LIME_NUM_SAMPLES
        )

        target_label = 1 if 1 in exp.available_labels() else exp.available_labels()[0]
        fmap_sorted = sorted(exp.as_map()[target_label], key=lambda t: abs(t[1]), reverse=True)

        row = df_samples[df_samples["sample_id"] == sid].iloc[0]
        for rank, (f_idx, w) in enumerate(fmap_sorted, start=1):
            fi = int(f_idx)
            long_records.append({
                "model": ens_name,
                "group": row["group"],
                "sample_id": int(sid),
                "y_true": int(row["y_true"]),
                "y_pred": int(row["y_pred"]),
                "p_phishing": float(row["p_phishing"]),
                "explained_label": int(target_label),
                "feature_idx": fi,
                "feature_name": feature_cols[fi],
                "weight": float(w),
                "abs_weight": float(abs(w)),
                "rank": int(rank)
            })

    df_long = pd.DataFrame(long_records)
    save_csv(out_long, df_long)
    save_csv(out_samples, df_samples)

    save_json(out_meta, {
        "ensemble": ens_name,
        "n_samples": int(len(sample_ids)),
        "num_features_saved": LIME_NUM_FEATURES,
        "num_samples_lime": LIME_NUM_SAMPLES,
        "lime_train_sample": int(min(LIME_TRAIN_SAMPLE, len(X_train_raw))),
        "seed": SEED
    })

    print(f"✅ {ens_name} LIME saved -> {out_dir}")

In [20]:
import shap
import matplotlib.pyplot as plt

def run_shap_global_cached(ens_name: str, ens_dir: Path, predict_proba_fn, bg_idx):
    out_dir = ens_dir / "global_shap"
    out_imp = out_dir / "shap_importance.csv"
    out_meta = out_dir / "shap_meta.json"
    out_sum_png = out_dir / "shap_summary.png"
    out_bar_png = out_dir / "shap_bar_top20.png"

    if exists(out_imp) and exists(out_meta):
        print(f"⏭️ {ens_name} Global SHAP cache var.")
        return

    feature_cols = list(feature_cols_global)

    # background + test sample
    bg = X_train_raw.iloc[bg_idx][feature_cols].values
    test_sample = X_test_raw[feature_cols].sample(
        n=min(GLOBAL_SHAP_TEST_SAMPLE, len(X_test_raw)),
        random_state=SEED
    ).values

    def predict_fn_np(z):
        z_df = pd.DataFrame(z, columns=feature_cols)
        return predict_proba_fn(z_df)

    explainer = shap.Explainer(predict_fn_np, bg)  # PermutationExplainer seçebilir
    exp = explainer(test_sample)

    sv = np.asarray(exp.values)
    if sv.ndim == 3 and sv.shape[2] >= 2:
        sv = sv[:, :, 1]  # class=1
    elif sv.ndim != 2:
        raise ValueError(f"Unexpected SHAP values shape: {sv.shape}")

    mean_abs = np.mean(np.abs(sv), axis=0)
    df_imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)

    save_csv(out_imp, df_imp)

    save_json(out_meta, {
        "ensemble": ens_name,
        "explainer": type(explainer).__name__,
        "bg_size": int(bg.shape[0]),
        "test_sample_size": int(test_sample.shape[0]),
        "sv_shape": list(np.asarray(exp.values).shape),
        "seed": SEED
    })

    # Grafikler
    if MAKE_PNG:
        shap.summary_plot(sv, features=test_sample, feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(out_sum_png, dpi=220)
        plt.close()

        top = df_imp.head(TOPK_GLOBAL)
        plt.figure(figsize=(8, 4))
        plt.bar(top["feature"], top["mean_abs_shap"])
        plt.xticks(rotation=45, ha="right")
        plt.title(f"{ens_name} Global SHAP Top-{TOPK_GLOBAL}")
        plt.tight_layout()
        plt.savefig(out_bar_png, dpi=220)
        plt.close()

    print(f"✅ {ens_name} Global SHAP saved -> {out_dir}")

def run_shap_local_cached(ens_name: str, ens_dir: Path, predict_proba_fn, df_samples: pd.DataFrame, bg_idx):
    out_dir = ens_dir / "local_shap"
    out_long = out_dir / "shap_local_long.csv"
    out_meta = out_dir / "shap_local_meta.json"

    if exists(out_long) and exists(out_meta):
        print(f"⏭️ {ens_name} Local SHAP cache var.")
        return

    feature_cols = list(feature_cols_global)

    bg = X_train_raw.iloc[bg_idx][feature_cols].values
    sample_ids = df_samples["sample_id"].astype(int).tolist()
    X_local = X_test_raw.iloc[sample_ids][feature_cols].values

    def predict_fn_np(z):
        z_df = pd.DataFrame(z, columns=feature_cols)
        return predict_proba_fn(z_df)

    explainer = shap.Explainer(predict_fn_np, bg)
    exp = explainer(X_local)

    sv = np.asarray(exp.values)
    if sv.ndim == 3 and sv.shape[2] >= 2:
        sv = sv[:, :, 1]
    elif sv.ndim != 2:
        raise ValueError(f"Unexpected SHAP values shape: {sv.shape}")

    long_records = []
    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    for row_i, sid in enumerate(sample_ids):
        vals = sv[row_i]
        abs_vals = np.abs(vals)
        top_idx = np.argsort(abs_vals)[::-1][:TOPK_LOCAL]

        for rank, fi in enumerate(top_idx, start=1):
            fi = int(fi)
            long_records.append({
                "model": ens_name,
                "sample_id": int(sid),
                "group": df_samples.loc[df_samples["sample_id"]==sid, "group"].values[0],
                "y_true": int(y_arr[sid]),
                "feature_name": feature_cols[fi],
                "shap_value": float(vals[fi]),
                "abs_shap": float(abs_vals[fi]),
                "rank": int(rank),
                "explainer": type(explainer).__name__
            })

    df_long = pd.DataFrame(long_records)
    save_csv(out_long, df_long)

    save_json(out_meta, {
        "ensemble": ens_name,
        "explainer": type(explainer).__name__,
        "bg_size": int(bg.shape[0]),
        "n_samples": int(len(sample_ids)),
        "topk_saved": TOPK_LOCAL,
        "sv_shape": list(np.asarray(exp.values).shape),
        "seed": SEED
    })

    print(f"✅ {ens_name} Local SHAP saved -> {out_dir}")

In [21]:
# feature_cols_global: veri kolonların (url + label hariç)
feature_cols_global = list(feature_cols)

def run_all_xai_for_ensemble(ens_name: str, predict_proba_fn, config: dict):
    ens_dir = ensure_ensemble_dir(ens_name, config)

    # bg idx cache
    bg_idx = get_or_make_bg_idx(ens_dir)

    # samples_400 cache
    df_samples = get_or_make_samples_400(ens_dir, predict_proba_fn)

    # LIME
    run_lime_cached(ens_name, ens_dir, predict_proba_fn, df_samples)

    # SHAP global + local
    run_shap_global_cached(ens_name, ens_dir, predict_proba_fn, bg_idx)
    run_shap_local_cached(ens_name, ens_dir, predict_proba_fn, df_samples, bg_idx)

    print(f"\n✅ DONE: {ens_name} (cache enabled)\n")
    return ens_dir

In [ ]:
ens_dir = run_all_xai_for_ensemble(
    ens_name="EnsembleSoft",
    predict_proba_fn=ensemble_soft_predict_proba,
    config={
        "type": "soft_voting",
        "models": list(pipelines.keys()),
        "seed": SEED,
        "bg_size": BG_SIZE,
        "global_shap_test_sample": GLOBAL_SHAP_TEST_SAMPLE,
        "lime_total": LIME_TOTAL
    }
)
print("EnsembleSoft dir:", ens_dir.resolve())

C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ samples_400 saved: outputs\ENSEMBLES\EnsembleSoft\samples_400.csv


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

✅ EnsembleSoft LIME saved -> outputs\ENSEMBLES\EnsembleSoft\lime


C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
PermutationExplainer explainer:   0%|                                                         | 1/3000 [00:00<?, ?it/s]C:\Users\elifo\anacond3\envs\url_ai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [1]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

DATA_PATH = "~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (579920, 77)


,url,label,url_length,domain_length,hostname_length,path_length,first_dir_length,tld_length,tld_length_domain,url_depth,...,whois_success,domain_age_days,expiration_days,creation_year,domain_is_recent,domain_registered_before_2020,registrar_valid,name_servers_count,is_privacy_protected,whois_missing
0,http://%20%25**)(**@fbrasil.com/old/lqjj0ukuvg...,1,57,7,24,26,3,3,3,3,...,1.0,5484.0,359.0,2010.0,0.0,1.0,1.0,2.0,0.0,0
1,https://*f003.backblazeb2.com*/file/pesosi/hom...,1,52,4,22,22,4,0,0,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,http://0-docusign-secured-843439-1-srs09.repli...,1,52,6,44,1,0,3,3,0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0
3,http://0-olx.1850943.xyz/,1,25,7,17,1,0,3,3,0,...,1.0,272.0,93.0,2025.0,1.0,0.0,1.0,2.0,0.0,0
4,http://0.0.0.0forum.cryptonight.net,1,35,11,28,0,0,3,3,0,...,1.0,475.0,254.0,2024.0,0.0,0.0,1.0,2.0,0.0,0


In [8]:
label_col = "label"

drop_cols = ["url"] if "url" in df.columns else []
feature_cols = [c for c in df.columns if c not in drop_cols + [label_col]]

X = df[feature_cols]
y = df[label_col]

print("Feature count:", len(feature_cols))

Feature count: 75


In [9]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print("Train:", X_train_raw.shape)
print("Test :", X_test_raw.shape)

Train: (463936, 75)
Test : (115984, 75)


In [10]:
import os, random, json
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -------- CONFIG (Asama0/Asama1 ile aynı mantık) --------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# DATA PATH (senin kullandığın)
DATA_PATH = os.path.expanduser("~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv")

df = pd.read_csv(DATA_PATH)
print("✅ Loaded:", df.shape)

label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"] if CONFIG["url_col"] in df.columns else None

drop_cols = [label_col]
if url_col:
    drop_cols.append(url_col)

feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[label_col].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["seed"],
    stratify=y
)

print("✅ Train:", X_train_raw.shape, " Test:", X_test_raw.shape)
print("✅ #Features:", len(feature_cols))

# -------- Base models (Asama0 baseline ayarların) --------
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=CONFIG["seed"]),
    "DecisionTree": DecisionTreeClassifier(random_state=CONFIG["seed"]),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG["seed"], n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=CONFIG["seed"],
        eval_metric="logloss",
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=63,
        random_state=CONFIG["seed"],
        n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800,
        learning_rate=0.05,
        depth=8,
        random_seed=CONFIG["seed"],
        verbose=False
    ),
}

# -------- Pipelines (imputer+scaler+model) --------
pipelines = {}
for name, model in base_models.items():
    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])
    pipe.fit(X_train_raw, y_train)
    pipelines[name] = pipe
    print("✅ Fitted:", name)

✅ Loaded: (579920, 77)
✅ Train: (463936, 75)  Test: (115984, 75)
✅ #Features: 75
✅ Fitted: LogReg
✅ Fitted: DecisionTree
✅ Fitted: RandomForest
✅ Fitted: XGBoost
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.068750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6109
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
✅ Fitted: LightGBM
✅ Fitted: CatBoost


In [11]:
import pandas as pd
from pathlib import Path

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
lime_samples_path = ENS_DIR / "lime" / "lime_samples.csv"
assert lime_samples_path.exists(), f"Bulunamadı: {lime_samples_path}"

df_samples = pd.read_csv(lime_samples_path)
sample_ids = df_samples["sample_id"].astype(int).tolist()

print("✅ LIME sample set loaded:", lime_samples_path)
print("n =", len(sample_ids))
print(df_samples["group"].value_counts())

✅ LIME sample set loaded: outputs\ENSEMBLES\EnsembleSoft\lime\lime_samples.csv
n = 400
group
correct    200
wrong      200
Name: count, dtype: int64


In [12]:
import numpy as np
import pandas as pd

def ensemble_soft_predict_proba(X_df):
    probs = []
    for name, pipe in pipelines.items():
        p = pipe.predict_proba(X_df)[:, 1]
        probs.append(p)
    mean_prob = np.vstack(probs).mean(axis=0)
    return np.column_stack([1 - mean_prob, mean_prob])

In [ ]:
import json, numpy as np
import shap
from pathlib import Path

def exists_ok(p: Path):
    return p.exists() and p.stat().st_size > 0

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
out_dir = ENS_DIR / "local_shap"
out_long = out_dir / "shap_local_long.csv"
out_meta = out_dir / "shap_local_meta.json"

bg_n = 200
topk_local = 30
SEED = 42

if exists_ok(out_long) and exists_ok(out_meta):
    print("⏭️ Local SHAP cache var, atlandı:", out_dir)
else:
    # Background index (varsa kullan, yoksa üret)
    bg_idx_path = ENS_DIR / "bg_idx.csv"
    if bg_idx_path.exists() and bg_idx_path.stat().st_size > 0:
        bg_idx = pd.read_csv(bg_idx_path)["idx"].astype(int).tolist()
    else:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(np.arange(len(X_train_raw)), size=min(2000, len(X_train_raw)), replace=False)
        pd.DataFrame({"idx": idx.astype(int)}).to_csv(bg_idx_path, index=False, encoding="utf-8")
        bg_idx = idx.tolist()

    feature_cols_list = list(feature_cols)

    bg = X_train_raw.iloc[bg_idx[:min(bg_n, len(bg_idx))]][feature_cols_list].values
    X_local = X_test_raw.iloc[sample_ids][feature_cols_list].values

    d = X_local.shape[1]
    max_evals = 2 * d + 1

    def predict_fn_np(z):
        z_df = pd.DataFrame(z, columns=feature_cols_list)
        return ensemble_soft_predict_proba(z_df)

    print(f"➡️ Local SHAP starting (same samples as LIME): n={X_local.shape[0]}, bg_n={bg.shape[0]}, d={d}, max_evals={max_evals}")
    explainer = shap.Explainer(predict_fn_np, bg)
    exp = explainer(X_local, max_evals=max_evals)

    sv_raw = np.asarray(exp.values)
    if sv_raw.ndim == 3 and sv_raw.shape[2] >= 2:
        sv = sv_raw[:, :, 1]
    elif sv_raw.ndim == 2:
        sv = sv_raw
    else:
        raise ValueError(f"Unexpected SHAP shape: {sv_raw.shape}")

    y_arr = y_test.values if hasattr(y_test, "values") else np.asarray(y_test)

    records = []
    for row_i, sid in enumerate(sample_ids):
        vals = sv[row_i]
        abs_vals = np.abs(vals)
        top_idx = np.argsort(abs_vals)[::-1][:topk_local]
        grp = df_samples.loc[df_samples["sample_id"] == sid, "group"].values[0]

        for rank, fi in enumerate(top_idx, start=1):
            fi = int(fi)
            records.append({
                "model": "EnsembleSoft",
                "sample_id": int(sid),
                "group": grp,
                "y_true": int(y_arr[sid]),
                "feature_name": feature_cols_list[fi],
                "shap_value": float(vals[fi]),
                "abs_shap": float(abs_vals[fi]),
                "rank": int(rank),
                "explainer": type(explainer).__name__
            })

    pd.DataFrame(records).to_csv(out_long, index=False, encoding="utf-8")
    save_json(out_meta, {
        "ensemble": "EnsembleSoft",
        "explainer": type(explainer).__name__,
        "bg_n": int(bg.shape[0]),
        "n_samples": int(len(sample_ids)),
        "topk_local": int(topk_local),
        "max_evals": int(max_evals),
        "seed": SEED,
        "sv_raw_shape": list(sv_raw.shape),
        "note": "Local SHAP computed on EXACT same sample_ids as LIME (lime_samples.csv)."
    })

    print("✅ EnsembleSoft Local SHAP saved ->", out_dir)

In [14]:
from pathlib import Path
Path("outputs/ENSEMBLES/EnsembleSoft/local_shap").mkdir(parents=True, exist_ok=True)
print("✅ created:", Path("outputs/ENSEMBLES/EnsembleSoft/local_shap").resolve())

✅ created: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs\ENSEMBLES\EnsembleSoft\local_shap


In [15]:
# Eğer records hala RAM'de duruyorsa:
pd.DataFrame(records).to_csv("outputs/ENSEMBLES/EnsembleSoft/local_shap/shap_local_long.csv", index=False, encoding="utf-8")
print("✅ saved shap_local_long.csv")

✅ saved shap_local_long.csv


In [16]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
meta_path = ENS_DIR / "local_shap" / "shap_local_meta.json"
csv_path  = ENS_DIR / "local_shap" / "shap_local_long.csv"

print("meta:", meta_path, "=>", meta_path.exists())
print("csv :", csv_path, "=>", csv_path.exists(), "size:", csv_path.stat().st_size if csv_path.exists() else None)

if not meta_path.exists():
    # minimum meta yazalım
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump({
            "ensemble": "EnsembleSoft",
            "note": "Local SHAP saved successfully. Sample ids are identical to lime_samples.csv."
        }, f, ensure_ascii=False, indent=2)
    print("✅ wrote:", meta_path)
else:
    print("✅ meta already exists")

meta: outputs\ENSEMBLES\EnsembleSoft\local_shap\shap_local_meta.json => False
csv : outputs\ENSEMBLES\EnsembleSoft\local_shap\shap_local_long.csv => True size: 1304466
✅ wrote: outputs\ENSEMBLES\EnsembleSoft\local_shap\shap_local_meta.json


In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

TOPK_COMPARE = 10
ENS_NAME = "EnsembleSoft"

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
COMPARE_DIR = Path("outputs/_shared/lime_shap_compare_400")
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

lime_long_path = ENS_DIR / "lime" / "lime_explanations_long.csv"
lime_samples_path = ENS_DIR / "lime" / "lime_samples.csv"
shap_long_path = ENS_DIR / "local_shap" / "shap_local_long.csv"

df_lime = pd.read_csv(lime_long_path)
df_samp = pd.read_csv(lime_samples_path)
df_shap = pd.read_csv(shap_long_path)

# LIME label filtresi (varsa)
if "explained_label" in df_lime.columns:
    df_lime = df_lime[df_lime["explained_label"] == 1].copy()

def topk_features(df, sid, sort_col, k):
    sub = df[df["sample_id"] == sid].copy()
    sub = sub.sort_values(sort_col, ascending=False).head(k)
    return sub

rows = []
for sid in df_samp["sample_id"].astype(int).tolist():
    lime_top = topk_features(df_lime, sid, "abs_weight" if "abs_weight" in df_lime.columns else "weight", TOPK_COMPARE)
    shap_top = topk_features(df_shap, sid, "abs_shap", TOPK_COMPARE)

    A = lime_top["feature_name"].tolist()
    B = shap_top["feature_name"].tolist()
    setA, setB = set(A), set(B)
    inter = setA & setB

    overlap = len(inter) / TOPK_COMPARE

    # Spearman: union üzerinde rank karşılaştır
    union = list(setA | setB)
    lime_rank = {f: (i+1) for i, f in enumerate(A)}
    shap_rank = {f: (i+1) for i, f in enumerate(B)}
    v1 = [lime_rank.get(f, TOPK_COMPARE+1) for f in union]
    v2 = [shap_rank.get(f, TOPK_COMPARE+1) for f in union]
    rho, _ = spearmanr(v1, v2)
    rho = 0.0 if pd.isna(rho) else float(rho)

    # sign agreement: ortak feature’larda işaret uyumu
    sign_agree = 0.0
    if len(inter) > 0:
        lime_map = lime_top.set_index("feature_name")
        shap_map = shap_top.set_index("feature_name")
        agrees = []
        for f in inter:
            s1 = np.sign(float(lime_map.loc[f, "weight"]))
            s2 = np.sign(float(shap_map.loc[f, "shap_value"]))
            agrees.append(1 if s1 == s2 else 0)
        sign_agree = float(np.mean(agrees))

    g = df_samp[df_samp["sample_id"] == sid].iloc[0]
    rows.append({
        "model": ENS_NAME,
        "sample_id": int(sid),
        "group": g["group"],
        "y_true": int(g["y_true"]),
        "y_pred": int(g["y_pred"]),
        "p_phishing": float(g["p_phishing"]),
        "topk": TOPK_COMPARE,
        "overlap_ratio": overlap,
        "spearman_rho": rho,
        "sign_agreement": sign_agree
    })

df_cmp = pd.DataFrame(rows)

per_sample_path = COMPARE_DIR / "lime_shap_compare_per_sample.csv"
if per_sample_path.exists():
    old = pd.read_csv(per_sample_path)
    old = old[old["model"] != ENS_NAME]
    df_all = pd.concat([old, df_cmp], ignore_index=True)
else:
    df_all = df_cmp

df_all.to_csv(per_sample_path, index=False, encoding="utf-8")

df_summary = (
    df_all.groupby(["model","group"])
    .agg(
        n=("sample_id","count"),
        overlap_mean=("overlap_ratio","mean"),
        overlap_std=("overlap_ratio","std"),
        rho_mean=("spearman_rho","mean"),
        rho_std=("spearman_rho","std"),
        sign_mean=("sign_agreement","mean"),
        sign_std=("sign_agreement","std"),
    )
    .reset_index()
)
summary_path = COMPARE_DIR / "lime_shap_compare_summary.csv"
df_summary.to_csv(summary_path, index=False, encoding="utf-8")

print("✅ Updated:", per_sample_path)
print("✅ Updated:", summary_path)
display(df_summary[df_summary["model"] == ENS_NAME])

✅ Updated: outputs\_shared\lime_shap_compare_400\lime_shap_compare_per_sample.csv
✅ Updated: outputs\_shared\lime_shap_compare_400\lime_shap_compare_summary.csv


,model,group,n,overlap_mean,overlap_std,rho_mean,rho_std,sign_mean,sign_std
4,EnsembleSoft,correct,200,0.285,0.093373,-0.296289,0.186657,0.995833,0.042393
5,EnsembleSoft,wrong,200,0.271,0.094358,-0.348903,0.165036,0.995833,0.042393


In [18]:
import shap, json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

ENS_NAME = "EnsembleSoft"
ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
steps = [200, 500, 1000]
bg_n = 200
SEED = 42

# bg_idx hazır olmalı
bg_idx_path = ENS_DIR / "bg_idx.csv"
bg_idx = pd.read_csv(bg_idx_path)["idx"].astype(int).tolist()

feature_cols_list = list(feature_cols)

def predict_fn_np(z):
    z_df = pd.DataFrame(z, columns=feature_cols_list)
    return ensemble_soft_predict_proba(z_df)

def compute_global_step(N):
    out_dir = ENS_DIR / "global_shap_steps" / f"N{N}"
    out_imp = out_dir / "shap_importance.csv"
    out_meta = out_dir / "shap_meta.json"
    out_dir.mkdir(parents=True, exist_ok=True)

    if out_imp.exists() and out_imp.stat().st_size > 0:
        return pd.read_csv(out_imp)

    bg = X_train_raw.iloc[bg_idx[:min(bg_n, len(bg_idx))]][feature_cols_list].values
    X = X_test_raw[feature_cols_list].sample(n=min(N, len(X_test_raw)), random_state=SEED).values

    d = X.shape[1]
    max_evals = 2 * d + 1

    print(f"➡️ Global SHAP N={X.shape[0]} starting (bg_n={bg.shape[0]}, max_evals={max_evals})")
    explainer = shap.Explainer(predict_fn_np, bg)
    exp = explainer(X, max_evals=max_evals)

    sv_raw = np.asarray(exp.values)
    if sv_raw.ndim == 3 and sv_raw.shape[2] >= 2:
        sv = sv_raw[:, :, 1]
    else:
        sv = sv_raw

    mean_abs = np.mean(np.abs(sv), axis=0)
    df_imp = pd.DataFrame({"feature": feature_cols_list, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
    df_imp.to_csv(out_imp, index=False, encoding="utf-8")

    with open(out_meta, "w", encoding="utf-8") as f:
        json.dump({
            "ensemble": ENS_NAME,
            "N": int(X.shape[0]),
            "bg_n": int(bg.shape[0]),
            "max_evals": int(max_evals),
            "seed": SEED,
            "explainer": type(explainer).__name__,
            "sv_raw_shape": list(np.asarray(exp.values).shape),
        }, f, ensure_ascii=False, indent=2)

    return df_imp

imps = {N: compute_global_step(N) for N in steps}
print("✅ global steps done/cached")

➡️ Global SHAP N=200 starting (bg_n=200, max_evals=151)


PermutationExplainer explainer: 201it [04:32,  1.42s/it]                                                               


➡️ Global SHAP N=500 starting (bg_n=200, max_evals=151)


PermutationExplainer explainer: 501it [10:01,  1.22s/it]                                                               


➡️ Global SHAP N=1000 starting (bg_n=200, max_evals=151)


PermutationExplainer explainer: 1001it [18:04,  1.09s/it]                                                              

✅ global steps done/cached


In [19]:
def topk_set(df, k=20):
    return set(df.head(k)["feature"].tolist())

def rank_map(df, k=100):
    top = df.head(k).reset_index(drop=True)
    return {row["feature"]: i+1 for i, row in top.iterrows()}

rows = []
for i in range(len(steps)):
    for j in range(i+1, len(steps)):
        a, b = steps[i], steps[j]
        A20, B20 = topk_set(imps[a], 20), topk_set(imps[b], 20)
        overlap20 = len(A20 & B20) / 20

        ra = rank_map(imps[a], 100)
        rb = rank_map(imps[b], 100)
        union = sorted(set(ra.keys()) | set(rb.keys()))
        va = [ra.get(f, 101) for f in union]
        vb = [rb.get(f, 101) for f in union]
        rho, _ = spearmanr(va, vb)

        rows.append({
            "N_a": a, "N_b": b,
            "top20_overlap": float(overlap20),
            "spearman_top100_union": float(rho)
        })

df_stability = pd.DataFrame(rows)
stab_path = ENS_DIR / "global_shap_stability.csv"
df_stability.to_csv(stab_path, index=False, encoding="utf-8")
print("✅ saved:", stab_path)
display(df_stability)

✅ saved: outputs\ENSEMBLES\EnsembleSoft\global_shap_stability.csv


,N_a,N_b,top20_overlap,spearman_top100_union
0,200,500,0.95,0.994452
1,200,1000,0.95,0.991835
2,500,1000,1.00,0.996871


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
ENS_DIR.mkdir(parents=True, exist_ok=True)

STEP_SAMPLE_PATH = ENS_DIR / "global_step_master_testset_1500.csv"

if STEP_SAMPLE_PATH.exists():
    df_master = pd.read_csv(STEP_SAMPLE_PATH)
    print("⏭️ 1500 master test set already exists.")
else:
    SEED = 42
    rng = np.random.RandomState(SEED)

    max_n = min(1500, len(X_test_raw))
    idx = rng.choice(np.arange(len(X_test_raw)), size=max_n, replace=False)

    df_master = pd.DataFrame({"test_index": idx})
    df_master.to_csv(STEP_SAMPLE_PATH, index=False, encoding="utf-8")

    print("✅ 1500 master test set created.")

print("Master size:", len(df_master))

✅ 1500 master test set created.
Master size: 1500


In [21]:
def compute_global_step_fixed(N):
    out_dir = ENS_DIR / "global_shap_steps_fixed" / f"N{N}"
    out_imp = out_dir / "shap_importance.csv"
    out_meta = out_dir / "shap_meta.json"
    out_dir.mkdir(parents=True, exist_ok=True)

    if out_imp.exists() and out_imp.stat().st_size > 0:
        return pd.read_csv(out_imp)

    df_master = pd.read_csv(ENS_DIR / "global_step_master_testset_1500.csv")
    test_indices = df_master["test_index"].astype(int).tolist()

    feature_cols_list = list(feature_cols)

    # Background
    bg_idx = pd.read_csv(ENS_DIR / "bg_idx.csv")["idx"].astype(int).tolist()
    bg = X_train_raw.iloc[bg_idx[:200]][feature_cols_list].values

    # Sabit 1500 setten ilk N örnek
    chosen_idx = test_indices[:N]
    X = X_test_raw.iloc[chosen_idx][feature_cols_list].values

    d = X.shape[1]
    max_evals = 2 * d + 1

    def predict_fn_np(z):
        z_df = pd.DataFrame(z, columns=feature_cols_list)
        return ensemble_soft_predict_proba(z_df)

    print(f"➡️ Global SHAP fixed step N={N} starting...")
    explainer = shap.Explainer(predict_fn_np, bg)
    exp = explainer(X, max_evals=max_evals)

    sv_raw = np.asarray(exp.values)
    if sv_raw.ndim == 3 and sv_raw.shape[2] >= 2:
        sv = sv_raw[:, :, 1]
    else:
        sv = sv_raw

    mean_abs = np.mean(np.abs(sv), axis=0)
    df_imp = pd.DataFrame({
        "feature": feature_cols_list,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)

    df_imp.to_csv(out_imp, index=False, encoding="utf-8")

    return df_imp

In [22]:
steps = [200, 500, 1000, 1500]

imps_fixed = {}
for N in steps:
    imps_fixed[N] = compute_global_step_fixed(N)

print("✅ All fixed-step global SHAP runs complete.")

➡️ Global SHAP fixed step N=200 starting...


PermutationExplainer explainer: 201it [03:44,  1.17s/it]                                                               


➡️ Global SHAP fixed step N=500 starting...


PermutationExplainer explainer: 501it [09:23,  1.15s/it]                                                               


➡️ Global SHAP fixed step N=1000 starting...


PermutationExplainer explainer: 1001it [19:02,  1.15s/it]                                                              


➡️ Global SHAP fixed step N=1500 starting...


PermutationExplainer explainer: 1501it [27:53,  1.12s/it]                                                              

✅ All fixed-step global SHAP runs complete.


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

ENS_DIR = Path("outputs/ENSEMBLES/EnsembleSoft")
STEPS_DIR = ENS_DIR / "global_shap_steps_fixed"
OUT_TABLE = ENS_DIR / "global_shap_stability_fixed_200_500_1000_1500.csv"
OUT_MATRIX = ENS_DIR / "global_shap_stability_fixed_matrix.csv"

# ---- Ayarlar
steps = [200, 500, 1000, 1500]
TOPK_OVERLAP = 20
TOPK_RANK = 100
MISS_RANK = TOPK_RANK + 1

# ---- Importance dosyalarını oku
imps = {}
for N in steps:
    p = STEPS_DIR / f"N{N}" / "shap_importance.csv"
    if not p.exists() or p.stat().st_size == 0:
        raise FileNotFoundError(f"Missing: {p}")
    imps[N] = pd.read_csv(p)

def topk_set(df, k=20):
    return set(df.head(k)["feature"].tolist())

def rank_map(df, k=100):
    top = df.head(k).reset_index(drop=True)
    return {row["feature"]: i+1 for i, row in top.iterrows()}

# ---- Pairwise tablo
rows = []
for i in range(len(steps)):
    for j in range(i+1, len(steps)):
        a, b = steps[i], steps[j]

        A = topk_set(imps[a], TOPK_OVERLAP)
        B = topk_set(imps[b], TOPK_OVERLAP)
        overlap = len(A & B) / TOPK_OVERLAP

        ra = rank_map(imps[a], TOPK_RANK)
        rb = rank_map(imps[b], TOPK_RANK)
        union = sorted(set(ra.keys()) | set(rb.keys()))

        va = [ra.get(f, MISS_RANK) for f in union]
        vb = [rb.get(f, MISS_RANK) for f in union]
        rho, _ = spearmanr(va, vb)
        rho = 0.0 if pd.isna(rho) else float(rho)

        rows.append({
            "N_a": a,
            "N_b": b,
            "top20_overlap": float(overlap),
            "spearman_top100_union": float(rho),
            "union_size_top100": int(len(union))
        })

df_stab = pd.DataFrame(rows).sort_values(["N_a", "N_b"])
df_stab.to_csv(OUT_TABLE, index=False, encoding="utf-8")
print("✅ Saved stability table:", OUT_TABLE)

# ---- Spearman matrix (heatmap için)
mat = pd.DataFrame(index=steps, columns=steps, dtype=float)
for a in steps:
    for b in steps:
        if a == b:
            mat.loc[a, b] = 1.0
        else:
            r = df_stab[((df_stab["N_a"] == min(a,b)) & (df_stab["N_b"] == max(a,b)))]
            mat.loc[a, b] = float(r["spearman_top100_union"].iloc[0])
mat.to_csv(OUT_MATRIX, encoding="utf-8")
print("✅ Saved spearman matrix:", OUT_MATRIX)

display(df_stab)

# ---- Grafik 1: Spearman heatmap (matshow)
plt.figure(figsize=(6, 5))
plt.imshow(mat.values, aspect="auto")
plt.xticks(range(len(steps)), steps)
plt.yticks(range(len(steps)), steps)
plt.colorbar()
plt.title("Global SHAP Stability (Spearman, Top-100 union)")
plt.tight_layout()
heatmap_path = ENS_DIR / "global_shap_stability_fixed_spearman_heatmap.png"
plt.savefig(heatmap_path, dpi=220)
plt.close()
print("✅ Saved heatmap:", heatmap_path)

# ---- Grafik 2: Overlap vs N (baseline: 1500 ile)
# 1500 ile her N'in overlap'ı (Top20)
base = 1500
over_rows = []
base_set = topk_set(imps[base], TOPK_OVERLAP)
for N in steps:
    sN = topk_set(imps[N], TOPK_OVERLAP)
    over_rows.append({"N": N, "top20_overlap_vs_1500": len(sN & base_set)/TOPK_OVERLAP})
df_over = pd.DataFrame(over_rows).sort_values("N")

plt.figure(figsize=(7, 4))
plt.plot(df_over["N"], df_over["top20_overlap_vs_1500"], marker="o")
plt.xticks(steps)
plt.ylim(0, 1.05)
plt.title("Top-20 Overlap vs 1500 (Fixed-step)")
plt.xlabel("N (test instances used for global SHAP)")
plt.ylabel("Top-20 overlap")
plt.tight_layout()
over_path = ENS_DIR / "global_shap_stability_fixed_top20_overlap_vs_1500.png"
plt.savefig(over_path, dpi=220)
plt.close()
print("✅ Saved overlap plot:", over_path)

✅ Saved stability table: outputs\ENSEMBLES\EnsembleSoft\global_shap_stability_fixed_200_500_1000_1500.csv
✅ Saved spearman matrix: outputs\ENSEMBLES\EnsembleSoft\global_shap_stability_fixed_matrix.csv


,N_a,N_b,top20_overlap,spearman_top100_union,union_size_top100
0,200,500,0.95,0.991067,75
1,200,1000,0.95,0.992319,75
2,200,1500,0.95,0.990669,75
3,500,1000,1.00,0.996387,75
4,500,1500,1.00,0.995078,75
5,1000,1500,1.00,0.998122,75


✅ Saved heatmap: outputs\ENSEMBLES\EnsembleSoft\global_shap_stability_fixed_spearman_heatmap.png
✅ Saved overlap plot: outputs\ENSEMBLES\EnsembleSoft\global_shap_stability_fixed_top20_overlap_vs_1500.png


In [27]:
import os, glob
import pandas as pd

# 1) outputs klasörün nerede?
ROOT = "."                 # gerekirse değiştir: r"C:\Users\...\Makale"
OUTPUTS_DIR = os.path.join(ROOT, "outputs")

print("OUTPUTS_DIR:", os.path.abspath(OUTPUTS_DIR))
print("Exists?:", os.path.exists(OUTPUTS_DIR))
print()

if not os.path.exists(OUTPUTS_DIR):
    raise FileNotFoundError("outputs klasörü bulunamadı. ROOT yolunu doğru ayarla.")

# 2) Kandidat dosya pattern'leri (master1500 / bg200 / split idx)
patterns = [
    "**/*master*1500*idx*.csv",
    "**/*master*idx*1500*.csv",
    "**/*master_test*1500*.csv",
    "**/*test*1500*idx*.csv",
    "**/*global*1500*idx*.csv",

    "**/*background*200*idx*.csv",
    "**/*bg*200*idx*.csv",
    "**/*background_200*.csv",
    "**/*bg_200*.csv",

    "**/*split*train*.csv",
    "**/*split*test*.csv",
    "**/*split_indices_train*.csv",
    "**/*split_indices_test*.csv",
    "**/*train_idx*.csv",
    "**/*test_idx*.csv",
]

found = []
for p in patterns:
    found += glob.glob(os.path.join(OUTPUTS_DIR, p), recursive=True)

# benzersizleştir ve sırala
found = sorted(set(found), key=lambda x: (0 if "_shared" in x else 1, x.lower()))

print(f"Found {len(found)} candidate index files:\n")
for f in found:
    size_kb = os.path.getsize(f) / 1024
    print(f"- {f}  ({size_kb:.1f} KB)")

# 3) Bulduklarımızın ilk 5 satırını göster (formatı kontrol)
print("\n--- Preview (first 5 rows) ---")
for f in found[:10]:  # ilk 10 dosyayı önizle
    try:
        df = pd.read_csv(f)
        cols = list(df.columns)
        print(f"\nFILE: {f}")
        print("cols:", cols)
        print(df.head())
    except Exception as e:
        print(f"\nFILE: {f}")
        print("Could not read CSV:", repr(e))

OUTPUTS_DIR: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Makale\outputs
Exists?: True

Found 3 candidate index files:

- .\outputs\_shared\split_indices_test.csv  (884.4 KB)
- .\outputs\_shared\split_indices_train.csv  (3537.7 KB)
- .\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv  (10.3 KB)

--- Preview (first 5 rows) ---

FILE: .\outputs\_shared\split_indices_test.csv
cols: ['test_idx']
   test_idx
0    235650
1    536480
2    530828
3    273597
4     51233

FILE: .\outputs\_shared\split_indices_train.csv
cols: ['train_idx']
   train_idx
0     482483
1     427217
2     297322
3     113283
4     398446

FILE: .\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv
cols: ['test_index']
   test_index
0       80656
1       23927
2       19864
3       79879
4       67408


In [28]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

SEED = 42
BG_N = 200

# Paths (senin çıktına göre net)
TRAIN_IDX_PATH = r".\outputs\_shared\split_indices_train.csv"
MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"
BG200_OUT_PATH = r".\outputs\_shared\background_200_idx.csv"

# 1) Var mı kontrol
print("TRAIN IDX:", TRAIN_IDX_PATH, "exists?", os.path.exists(TRAIN_IDX_PATH))
print("MASTER1500:", MASTER1500_PATH, "exists?", os.path.exists(MASTER1500_PATH))
print("BG200 OUT:", BG200_OUT_PATH, "exists?", os.path.exists(BG200_OUT_PATH))

# 2) Master 1500 oku (sadece kontrol amaçlı)
master_df = pd.read_csv(MASTER1500_PATH)
master_col = master_df.columns[0]  # 'test_index'
master1500_idx = master_df[master_col].to_numpy()
print("\nMaster1500 size:", len(master1500_idx), "col:", master_col)

# 3) Background yoksa üret (train havuzundan stratified)
if os.path.exists(BG200_OUT_PATH):
    bg_df = pd.read_csv(BG200_OUT_PATH)
    print("\n✅ BG200 already exists:", BG200_OUT_PATH)
    print(bg_df.head())
else:
    # Aşama2’de y zaten var olmalı. Eğer yoksa df->y çıkar.
    if "y" not in globals():
        if "df" in globals() and "label" in df.columns:
            y = df["label"].astype(int)
        else:
            raise RuntimeError("y bulunamadı. Aşama2’de y (label) değişkeni hazır olmalı ya da df içinde label olmalı.")

    train_df = pd.read_csv(TRAIN_IDX_PATH)
    train_col = train_df.columns[0]  # 'train_idx'
    train_idx = train_df[train_col].to_numpy()
    print("\nTrain pool size:", len(train_idx), "col:", train_col)

    if BG_N > len(train_idx):
        raise ValueError("BG_N train havuzundan büyük olamaz.")

    # stratified pick
    y_pool = y.iloc[train_idx].values
    sss = StratifiedShuffleSplit(n_splits=1, train_size=BG_N, random_state=SEED)
    sel, _ = next(sss.split(train_idx, y_pool))
    bg_idx = train_idx[sel]

    out = pd.DataFrame({"bg_idx": bg_idx})
    out.to_csv(BG200_OUT_PATH, index=False)
    print("\n✅ BG200 created and saved:", BG200_OUT_PATH)
    print(out.head())

TRAIN IDX: .\outputs\_shared\split_indices_train.csv exists? True
MASTER1500: .\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv exists? True
BG200 OUT: .\outputs\_shared\background_200_idx.csv exists? False

Master1500 size: 1500 col: test_index

Train pool size: 463936 col: train_idx

✅ BG200 created and saved: .\outputs\_shared\background_200_idx.csv
   bg_idx
0  339832
1  157062
2  265277
3   40251
4  457595


In [30]:
[x for x in globals().keys() if "pipe" in x.lower() or "model" in x.lower()]

['Pipeline',
 'base_models',
 'pipelines',
 'model',
 'pipe',
 'build_pipelines',
 'run_model_global_fixedstep']

In [31]:
import inspect
from sklearn.utils.validation import check_is_fitted

print("pipelines type:", type(pipelines))

# dict değilse, içeriği anlamaya çalışalım
if isinstance(pipelines, dict):
    print("pipelines keys:", list(pipelines.keys()))
    print("\n--- Pipeline fit status ---")
    for k, p in pipelines.items():
        try:
            # son adım classifier
            last_step_name, last_est = list(p.named_steps.items())[-1]
            check_is_fitted(last_est)
            status = "FITTED ✅"
        except Exception:
            status = "NOT FITTED ❌"
        print(f"{k:10s} -> {type(p).__name__} | last_step={last_step_name} ({type(last_est).__name__}) | {status}")
else:
    # dict değilse ilk 200 karakterini bas
    print("pipelines preview:", str(pipelines)[:200])

print("\npipe type:", type(pipe))
try:
    last_step_name, last_est = list(pipe.named_steps.items())[-1]
    check_is_fitted(last_est)
    print("pipe FITTED ✅ | last_step:", last_step_name, type(last_est).__name__)
except Exception as e:
    print("pipe NOT FITTED ❌", repr(e))

pipelines type: <class 'dict'>
pipelines keys: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']

--- Pipeline fit status ---
LogReg     -> Pipeline | last_step=model (LogisticRegression) | FITTED ✅
DecisionTree -> Pipeline | last_step=model (DecisionTreeClassifier) | FITTED ✅
RandomForest -> Pipeline | last_step=model (RandomForestClassifier) | FITTED ✅
XGBoost    -> Pipeline | last_step=model (XGBClassifier) | FITTED ✅
LightGBM   -> Pipeline | last_step=model (LGBMClassifier) | FITTED ✅
CatBoost   -> Pipeline | last_step=model (CatBoostClassifier) | FITTED ✅

pipe type: <class 'sklearn.pipeline.Pipeline'>
pipe FITTED ✅ | last_step: model CatBoostClassifier


In [32]:
import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
import joblib

SEED = 42
STEPS = [200, 500, 1000, 1500]
TOPK = 30

MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"
BG200_PATH      = r".\outputs\_shared\background_200_idx.csv"
OUT_ROOT        = r".\outputs"

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_json(path, obj):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# 0) X/y yoksa df'den türet
if "X" not in globals() or "y" not in globals():
    if "df" in globals() and "label" in df.columns:
        y = df["label"].astype(int)
        drop_cols = [c for c in ["label", "url"] if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
    else:
        raise RuntimeError("X/y bulunamadı. Aşama2’de X ve y veya df(label) olmalı.")

# 1) pipelines dict zorunlu
if "pipelines" not in globals() or not isinstance(pipelines, dict) or len(pipelines) == 0:
    raise RuntimeError("pipelines dict bulunamadı. Şu an sende var ve FITTED görünüyordu; yeniden kontrol et.")

# 2) Indeksleri oku
m = pd.read_csv(MASTER1500_PATH)
master_idx = m[m.columns[0]].to_numpy()  # 'test_index'
bg = pd.read_csv(BG200_PATH)
bg_idx = bg[bg.columns[0]].to_numpy()    # 'bg_idx'

X_master = X.iloc[master_idx].copy()     # sıra Soft'un kullandığı sıradır
X_bg     = X.iloc[bg_idx].copy()

feature_names = list(X.columns)
d = len(feature_names)
max_evals = int(2*d + 1)

print("✅ MASTER1500:", MASTER1500_PATH, "n=", len(master_idx))
print("✅ BG200:", BG200_PATH, "n=", len(bg_idx))
print("Features:", d, "max_evals:", max_evals)
print("Models:", list(pipelines.keys()))

def compute_step(pipe, n, out_dir, model_name):
    step_csv = os.path.join(out_dir, f"shap_importance_N{n}.csv")
    step_npz = os.path.join(out_dir, f"shap_values_N{n}.npz")

    # resume-safe: varsa skip
    if os.path.exists(step_csv):
        print(f"[{model_name}] ✅ exists -> N={n} (skip)")
        return

    X_step = X_master.iloc[:n].copy()

    def f(Xinput):
        return pipe.predict_proba(Xinput)[:, 1]

    explainer = shap.Explainer(f, X_bg, feature_names=feature_names)
    sv = explainer(X_step, max_evals=max_evals)

    values = sv.values
    if values.ndim != 2:
        values = np.array(values).reshape(values.shape[0], -1)

    imp = np.mean(np.abs(values), axis=0)

    imp_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": imp})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp_df["rank"] = np.arange(1, len(imp_df) + 1)

    imp_df.to_csv(step_csv, index=False)
    np.savez_compressed(step_npz, shap_values=values)

    print(f"[{model_name}] ✅ computed -> N={n}")

# 3) Her model için çalıştır + kaydet
for model_name, pipe in pipelines.items():
    out_dir = os.path.join(OUT_ROOT, model_name, "global_shap_fixedstep_1500_200")
    ensure_dir(out_dir)

    # modeli de kaydedelim (Jupyter kapanınca da dursun)
    model_path = os.path.join(OUT_ROOT, model_name, "model_pipeline.joblib")
    if not os.path.exists(model_path):
        joblib.dump(pipe, model_path)

    meta_path = os.path.join(out_dir, "meta.json")
    if not os.path.exists(meta_path):
        meta = {
            "model_name": model_name,
            "seed": SEED,
            "steps": STEPS,
            "topk": TOPK,
            "max_evals": max_evals,
            "master1500_file": MASTER1500_PATH,
            "bg200_file": BG200_PATH,
            "feature_count": d,
            "feature_names": feature_names,
            "predict_interface": "pipe.predict_proba(X)[:,1]",
            "note": "Master1500 order taken from Soft ensemble file; fixed-step uses first N rows of this order."
        }
        save_json(meta_path, meta)

    # fixed steps
    for n in STEPS:
        compute_step(pipe, n, out_dir, model_name)

    # merged table
    dfs = []
    for n in STEPS:
        df_n = pd.read_csv(os.path.join(out_dir, f"shap_importance_N{n}.csv"))[["feature", "mean_abs_shap"]]
        df_n = df_n.rename(columns={"mean_abs_shap": f"N{n}"})
        dfs.append(df_n)

    merged = dfs[0]
    for df_n in dfs[1:]:
        merged = merged.merge(df_n, on="feature", how="left")
    merged.to_csv(os.path.join(out_dir, "shap_importance_steps_merged.csv"), index=False)

    # topK for N=1500
    top1500 = pd.read_csv(os.path.join(out_dir, "shap_importance_N1500.csv"))
    top1500.head(TOPK).to_csv(os.path.join(out_dir, f"top{TOPK}_features_N1500.csv"), index=False)

print("\n=== DONE: All models global SHAP fixed-step saved ===")

✅ MASTER1500: .\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv n= 1500
✅ BG200: .\outputs\_shared\background_200_idx.csv n= 200
Features: 75 max_evals: 151
Models: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
[LogReg] ✅ computed -> N=200


PermutationExplainer explainer: 501it [00:15, 10.90it/s]                                                               


[LogReg] ✅ computed -> N=500


PermutationExplainer explainer: 1001it [00:30, 21.44it/s]                                                              


[LogReg] ✅ computed -> N=1000


...

PermutationExplainer explainer: 201it [01:21,  2.15it/s]                                                               


[RandomForest] ✅ computed -> N=200


PermutationExplainer explainer: 501it [03:08,  2.54it/s]                                                               


[RandomForest] ✅ computed -> N=500


KeyboardInterrupt: 

In [33]:
import pandas as pd

MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"

m = pd.read_csv(MASTER1500_PATH)
master_idx = m[m.columns[0]].to_numpy()

# y değişkeni zaten var
y_master = y.iloc[master_idx]

print("Master1500 class distribution:")
print(y_master.value_counts(normalize=True))

Master1500 class distribution:
label
1    0.529333
0    0.470667
Name: proportion, dtype: float64


In [34]:
import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
import joblib

SEED = 42
STEPS = [200, 500, 1000, 1500]
TOPK = 30

MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"
BG200_PATH      = r".\outputs\_shared\background_200_idx.csv"
OUT_ROOT        = r".\outputs"

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_json(path, obj):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# 0) X/y yoksa df'den türet
if "X" not in globals() or "y" not in globals():
    if "df" in globals() and "label" in df.columns:
        y = df["label"].astype(int)
        drop_cols = [c for c in ["label", "url"] if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
    else:
        raise RuntimeError("X/y bulunamadı. Aşama2’de X ve y veya df(label) olmalı.")

# 1) pipelines dict zorunlu
if "pipelines" not in globals() or not isinstance(pipelines, dict) or len(pipelines) == 0:
    raise RuntimeError("pipelines dict bulunamadı. Şu an sende var ve FITTED görünüyordu; yeniden kontrol et.")

# 2) Indeksleri oku
m = pd.read_csv(MASTER1500_PATH)
master_idx = m[m.columns[0]].to_numpy()  # 'test_index'
bg = pd.read_csv(BG200_PATH)
bg_idx = bg[bg.columns[0]].to_numpy()    # 'bg_idx'

X_master = X.iloc[master_idx].copy()     # sıra Soft'un kullandığı sıradır
X_bg     = X.iloc[bg_idx].copy()

feature_names = list(X.columns)
d = len(feature_names)
max_evals = int(2*d + 1)

print("✅ MASTER1500:", MASTER1500_PATH, "n=", len(master_idx))
print("✅ BG200:", BG200_PATH, "n=", len(bg_idx))
print("Features:", d, "max_evals:", max_evals)
print("Models:", list(pipelines.keys()))

def compute_step(pipe, n, out_dir, model_name):
    step_csv = os.path.join(out_dir, f"shap_importance_N{n}.csv")
    step_npz = os.path.join(out_dir, f"shap_values_N{n}.npz")

    # resume-safe: varsa skip
    if os.path.exists(step_csv):
        print(f"[{model_name}] ✅ exists -> N={n} (skip)")
        return

    X_step = X_master.iloc[:n].copy()

    def f(Xinput):
        return pipe.predict_proba(Xinput)[:, 1]

    explainer = shap.Explainer(f, X_bg, feature_names=feature_names)
    sv = explainer(X_step, max_evals=max_evals)

    values = sv.values
    if values.ndim != 2:
        values = np.array(values).reshape(values.shape[0], -1)

    imp = np.mean(np.abs(values), axis=0)

    imp_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": imp})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp_df["rank"] = np.arange(1, len(imp_df) + 1)

    imp_df.to_csv(step_csv, index=False)
    np.savez_compressed(step_npz, shap_values=values)

    print(f"[{model_name}] ✅ computed -> N={n}")

# 3) Her model için çalıştır + kaydet
for model_name, pipe in pipelines.items():
    out_dir = os.path.join(OUT_ROOT, model_name, "global_shap_fixedstep_1500_200")
    ensure_dir(out_dir)

    # modeli de kaydedelim (Jupyter kapanınca da dursun)
    model_path = os.path.join(OUT_ROOT, model_name, "model_pipeline.joblib")
    if not os.path.exists(model_path):
        joblib.dump(pipe, model_path)

    meta_path = os.path.join(out_dir, "meta.json")
    if not os.path.exists(meta_path):
        meta = {
            "model_name": model_name,
            "seed": SEED,
            "steps": STEPS,
            "topk": TOPK,
            "max_evals": max_evals,
            "master1500_file": MASTER1500_PATH,
            "bg200_file": BG200_PATH,
            "feature_count": d,
            "feature_names": feature_names,
            "predict_interface": "pipe.predict_proba(X)[:,1]",
            "note": "Master1500 order taken from Soft ensemble file; fixed-step uses first N rows of this order."
        }
        save_json(meta_path, meta)

    # fixed steps
    for n in STEPS:
        compute_step(pipe, n, out_dir, model_name)

    # merged table
    dfs = []
    for n in STEPS:
        df_n = pd.read_csv(os.path.join(out_dir, f"shap_importance_N{n}.csv"))[["feature", "mean_abs_shap"]]
        df_n = df_n.rename(columns={"mean_abs_shap": f"N{n}"})
        dfs.append(df_n)

    merged = dfs[0]
    for df_n in dfs[1:]:
        merged = merged.merge(df_n, on="feature", how="left")
    merged.to_csv(os.path.join(out_dir, "shap_importance_steps_merged.csv"), index=False)

    # topK for N=1500
    top1500 = pd.read_csv(os.path.join(out_dir, "shap_importance_N1500.csv"))
    top1500.head(TOPK).to_csv(os.path.join(out_dir, f"top{TOPK}_features_N1500.csv"), index=False)

print("\n=== DONE: All models global SHAP fixed-step saved ===")

✅ MASTER1500: .\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv n= 1500
✅ BG200: .\outputs\_shared\background_200_idx.csv n= 200
Features: 75 max_evals: 151
Models: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
[LogReg] ✅ exists -> N=200 (skip)
[LogReg] ✅ exists -> N=500 (skip)
[LogReg] ✅ exists -> N=1000 (skip)
[LogReg] ✅ exists -> N=1500 (skip)
[DecisionTree] ✅ exists -> N=200 (skip)
[DecisionTree] ✅ exists -> N=500 (skip)
[DecisionTree] ✅ exists -> N=1000 (skip)
[DecisionTree] ✅ exists -> N=1500 (skip)
[RandomForest] ✅ exists -> N=200 (skip)
[RandomForest] ✅ exists -> N=500 (skip)


PermutationExplainer explainer: 1001it [05:58,  2.71it/s]                                                              


[RandomForest] ✅ computed -> N=1000


PermutationExplainer explainer: 1501it [09:24,  2.62it/s]                                                              


[RandomForest] ✅ computed -> N=1500


...

[CatBoost] ✅ computed -> N=500


PermutationExplainer explainer: 1001it [05:30,  2.93it/s]                                                              


[CatBoost] ✅ computed -> N=1000


PermutationExplainer explainer: 1501it [09:03,  2.72it/s]                                                              

[CatBoost] ✅ computed -> N=1500

=== DONE: All models global SHAP fixed-step saved ===


In [35]:
# ============================================================
# ENSEMBLE HARD VOTING: Global SHAP fixed-step + Local SHAP + LIME
# Resume-safe: if output files exist -> skip
# Uses SAME master1500 (Soft file) + SAME bg200 (shared file)
# Tries to reuse existing lime_samples.csv if found; else creates and saves once.
# ============================================================

import os, glob, json, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import shap
import joblib

# LIME
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import StratifiedShuffleSplit

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
STEPS = [200, 500, 1000, 1500]
TOPK = 30
LOCAL_N_TOTAL = 400   # 200 correct + 200 wrong (if we have enough)
LOCAL_TOP_FEATURES = 10

OUT_ROOT = r".\outputs"
ENS_NAME = "EnsembleHard"
ENS_DIR  = os.path.join(OUT_ROOT, "ENSEMBLES", ENS_NAME)

MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"  # existing
BG200_PATH      = r".\outputs\_shared\background_200_idx.csv"                              # created earlier
TEST_IDX_PATH   = r".\outputs\_shared\split_indices_test.csv"
TRAIN_IDX_PATH  = r".\outputs\_shared\split_indices_train.csv"

# ----------------------------
# Sanity checks
# ----------------------------
os.makedirs(ENS_DIR, exist_ok=True)

if "pipelines" not in globals() or not isinstance(pipelines, dict) or len(pipelines) == 0:
    raise RuntimeError("pipelines dict bulunamadı. Aşama2’de fitted pipelines olmalı.")

# X,y yoksa df'den türet
if "X" not in globals() or "y" not in globals():
    if "df" in globals() and "label" in df.columns:
        y = df["label"].astype(int)
        drop_cols = [c for c in ["label", "url"] if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
    else:
        raise RuntimeError("X/y bulunamadı. Aşama2’de X,y veya df(label) olmalı.")

feature_names = list(X.columns)
d = len(feature_names)
max_evals = int(2*d + 1)

print("✅ Feature count:", d, "| max_evals:", max_evals)
print("✅ Base models:", list(pipelines.keys()))

# ----------------------------
# Helpers
# ----------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_json(path, obj):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def find_first(patterns, base_dir=OUT_ROOT):
    candidates = []
    for pat in patterns:
        candidates += glob.glob(os.path.join(base_dir, "**", pat), recursive=True)
    candidates = sorted(set(candidates))
    return candidates[0] if candidates else None

def read_idx_csv(path):
    df = pd.read_csv(path)
    return df[df.columns[0]].to_numpy()

# ----------------------------
# Load indices (NO re-split)
# ----------------------------
master_idx = read_idx_csv(MASTER1500_PATH)  # 'test_index'
bg_idx     = read_idx_csv(BG200_PATH)       # 'bg_idx'
test_idx   = read_idx_csv(TEST_IDX_PATH)    # 'test_idx'
train_idx  = read_idx_csv(TRAIN_IDX_PATH)   # 'train_idx'

X_master = X.iloc[master_idx].copy()  # order fixed by Soft master file
X_bg     = X.iloc[bg_idx].copy()
X_test   = X.iloc[test_idx].copy()
y_test   = y.iloc[test_idx].copy()
X_train  = X.iloc[train_idx].copy()
y_train  = y.iloc[train_idx].copy()

print("✅ MASTER1500 n =", len(master_idx), "| BG200 n =", len(bg_idx))
print("✅ Test pool n =", len(test_idx), "| Train pool n =", len(train_idx))

# ----------------------------
# HARD VOTING ensemble predict proba
# proba = fraction of base models voting class 1 (based on their own proba>=0.5)
# ----------------------------
base_model_names = list(pipelines.keys())

def hard_ensemble_proba_from_df(X_df: pd.DataFrame) -> np.ndarray:
    # returns p(y=1) as 1D array
    votes = []
    for name in base_model_names:
        p = pipelines[name].predict_proba(X_df)[:, 1]
        vote = (p >= 0.5).astype(int)
        votes.append(vote)
    votes = np.stack(votes, axis=1)  # (n, m)
    return votes.mean(axis=1)        # fraction

def hard_ensemble_predict_proba(X_df: pd.DataFrame) -> np.ndarray:
    p1 = hard_ensemble_proba_from_df(X_df)
    p0 = 1.0 - p1
    return np.column_stack([p0, p1])

# For SHAP we use f(X)->p1
def f_shap(Xinput):
    # Xinput may be numpy or df depending on explainer calls
    if isinstance(Xinput, np.ndarray):
        Xdf = pd.DataFrame(Xinput, columns=feature_names)
    else:
        Xdf = Xinput
    return hard_ensemble_proba_from_df(Xdf)

# ----------------------------
# Save ensemble meta once
# ----------------------------
meta_path = os.path.join(ENS_DIR, "meta.json")
if not os.path.exists(meta_path):
    meta = {
        "ensemble_type": "HardVoting",
        "seed": SEED,
        "base_models": base_model_names,
        "predict_interface": "vote_fraction from base pipelines using proba>=0.5",
        "master1500_file": MASTER1500_PATH,
        "bg200_file": BG200_PATH,
        "fixed_steps": STEPS,
        "max_evals": max_evals,
        "feature_count": d,
        "local_n_total": LOCAL_N_TOTAL,
        "local_top_features": LOCAL_TOP_FEATURES
    }
    save_json(meta_path, meta)

# ============================================================
# (1) GLOBAL SHAP fixed-step (resume-safe)
# ============================================================
GLOBAL_DIR = os.path.join(ENS_DIR, "global_shap_fixedstep_1500_200")
ensure_dir(GLOBAL_DIR)

def compute_global_step(n):
    step_csv = os.path.join(GLOBAL_DIR, f"shap_importance_N{n}.csv")
    step_npz = os.path.join(GLOBAL_DIR, f"shap_values_N{n}.npz")

    if os.path.exists(step_csv):
        print(f"[Global SHAP] ✅ exists -> N={n} (skip)")
        return

    X_step = X_master.iloc[:n].copy()
    explainer = shap.Explainer(f_shap, X_bg, feature_names=feature_names)

    sv = explainer(X_step, max_evals=max_evals)
    values = sv.values
    if values.ndim != 2:
        values = np.array(values).reshape(values.shape[0], -1)

    imp = np.mean(np.abs(values), axis=0)
    imp_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": imp})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp_df["rank"] = np.arange(1, len(imp_df) + 1)

    imp_df.to_csv(step_csv, index=False)
    np.savez_compressed(step_npz, shap_values=values)
    print(f"[Global SHAP] ✅ computed -> N={n}")

print("\n=== (1) Global SHAP fixed-step ===")
for n in STEPS:
    compute_global_step(n)

# merge table (resume-safe overwrite is OK)
dfs = []
for n in STEPS:
    df_n = pd.read_csv(os.path.join(GLOBAL_DIR, f"shap_importance_N{n}.csv"))[["feature", "mean_abs_shap"]]
    df_n = df_n.rename(columns={"mean_abs_shap": f"N{n}"})
    dfs.append(df_n)
merged = dfs[0]
for df_n in dfs[1:]:
    merged = merged.merge(df_n, on="feature", how="left")
merged.to_csv(os.path.join(GLOBAL_DIR, "shap_importance_steps_merged.csv"), index=False)

top1500 = pd.read_csv(os.path.join(GLOBAL_DIR, "shap_importance_N1500.csv"))
top1500.head(TOPK).to_csv(os.path.join(GLOBAL_DIR, f"top{TOPK}_features_N1500.csv"), index=False)

# ============================================================
# (2) LOCAL SAMPLES: reuse existing lime_samples if possible (resume-safe)
# ============================================================
LOCAL_DIR = os.path.join(ENS_DIR, "local_xai_400")
ensure_dir(LOCAL_DIR)

# Try to find an existing lime_samples.csv anywhere under outputs
existing_lime_samples = find_first([
    "*lime_samples*.csv",
    "*local_samples*.csv",
], base_dir=OUT_ROOT)

lime_samples_path = os.path.join(LOCAL_DIR, "lime_samples_400.csv")
lime_samples_meta = os.path.join(LOCAL_DIR, "lime_samples_meta.json")

def create_local_samples_400():
    """
    Creates a 400-sample set: 200 correct + 200 wrong (if available) using HARD ensemble on test set.
    Saves indices relative to original dataset (row indices).
    """
    # Predict on full test pool
    p_test = hard_ensemble_proba_from_df(X_test)
    y_pred = (p_test >= 0.5).astype(int)
    y_true = y_test.values

    correct_mask = (y_pred == y_true)
    wrong_mask = ~correct_mask

    correct_idx_pool = test_idx[correct_mask]
    wrong_idx_pool   = test_idx[wrong_mask]

    rng = np.random.RandomState(SEED)

    n_each = LOCAL_N_TOTAL // 2
    chosen = []

    # sample correct
    if len(correct_idx_pool) >= n_each:
        rng.shuffle(correct_idx_pool)
        chosen_correct = correct_idx_pool[:n_each]
    else:
        chosen_correct = correct_idx_pool  # take all if insufficient

    # sample wrong
    if len(wrong_idx_pool) >= n_each:
        rng.shuffle(wrong_idx_pool)
        chosen_wrong = wrong_idx_pool[:n_each]
    else:
        chosen_wrong = wrong_idx_pool

    chosen = np.concatenate([chosen_correct, chosen_wrong])
    rng.shuffle(chosen)

    # Label group info
    group = []
    chosen_set = set(chosen)
    for idx_ in chosen:
        if idx_ in set(chosen_correct):
            group.append("correct")
        else:
            group.append("wrong")

    out = pd.DataFrame({"row_idx": chosen, "group": group})
    out.to_csv(lime_samples_path, index=False)

    meta = {
        "created_by": "HardVoting ensemble",
        "seed": SEED,
        "strategy": "200 correct + 200 wrong (if available)",
        "test_pool_size": int(len(test_idx)),
        "n_correct_pool": int(correct_mask.sum()),
        "n_wrong_pool": int(wrong_mask.sum()),
        "n_selected": int(len(out)),
        "note": "If a previous lime_samples file existed, it was not used; this is a new set."
    }
    save_json(lime_samples_meta, meta)

    print("✅ Local sample set CREATED:", lime_samples_path, "| n=", len(out))
    return out

# Decide reuse vs create
local_samples_df = None

if os.path.exists(lime_samples_path):
    local_samples_df = pd.read_csv(lime_samples_path)
    print("\n✅ Reusing existing local samples (already in EnsembleHard folder):", lime_samples_path)

else:
    # Try to reuse existing samples from previous experiments
    if existing_lime_samples is not None:
        try:
            tmp = pd.read_csv(existing_lime_samples)
            # Accept common column names
            if "row_idx" in tmp.columns:
                col = "row_idx"
            elif "idx" in tmp.columns:
                col = "idx"
            elif "test_index" in tmp.columns:
                col = "test_index"
            elif "test_idx" in tmp.columns:
                col = "test_idx"
            elif "sample_id" in tmp.columns:
                col = "sample_id"
            else:
                col = tmp.columns[0]

            # If file has more than 400, take first 400 deterministically
            vals = tmp[col].to_numpy()
            vals = vals[:LOCAL_N_TOTAL] if len(vals) >= LOCAL_N_TOTAL else vals

            local_samples_df = pd.DataFrame({"row_idx": vals})
            local_samples_df.to_csv(lime_samples_path, index=False)

            meta = {
                "reused_from": existing_lime_samples,
                "used_column": col,
                "n_source": int(len(tmp)),
                "n_used": int(len(local_samples_df)),
                "note": "Reused previous sample set to enable direct comparisons."
            }
            save_json(lime_samples_meta, meta)

            print("\n✅ Reused previous sample file:", existing_lime_samples)
            print("   Saved as:", lime_samples_path, "| n=", len(local_samples_df))
        except Exception as e:
            print("\n⚠️ Could not reuse previous lime_samples file due to:", repr(e))
            local_samples_df = create_local_samples_400()
    else:
        print("\n⚠️ No previous lime_samples/local_samples file found under outputs/. Creating a new one.")
        local_samples_df = create_local_samples_400()

# Build local X/y
local_idx = local_samples_df["row_idx"].to_numpy()
X_local = X.iloc[local_idx].copy()
y_local = y.iloc[local_idx].copy()

# ============================================================
# (3) LOCAL SHAP on same local samples (resume-safe)
# ============================================================
LOCAL_SHAP_DIR = os.path.join(LOCAL_DIR, "local_shap")
ensure_dir(LOCAL_SHAP_DIR)

local_shap_long_path = os.path.join(LOCAL_SHAP_DIR, "shap_local_long.csv")
local_shap_meta_path = os.path.join(LOCAL_SHAP_DIR, "shap_local_meta.json")

def local_shap_to_long(shap_values, row_indices, feature_names):
    # returns long df: row_idx, feature, shap_value, abs_shap, rank_abs, sign
    abs_vals = np.abs(shap_values)
    ranks = (-abs_vals).argsort(axis=1)  # indices sorted by abs desc

    rows = []
    for i in range(shap_values.shape[0]):
        idx_ = row_indices[i]
        # compute rank per feature: we can compute rank positions
        rank_pos = np.empty(shap_values.shape[1], dtype=int)
        rank_pos[ranks[i]] = np.arange(1, shap_values.shape[1] + 1)

        for j, fn in enumerate(feature_names):
            v = shap_values[i, j]
            rows.append((idx_, fn, float(v), float(abs(v)), int(rank_pos[j]), 1 if v >= 0 else -1))
    return pd.DataFrame(rows, columns=["row_idx", "feature", "shap_value", "abs_shap", "rank_abs", "sign"])

print("\n=== (3) Local SHAP (same samples) ===")
if os.path.exists(local_shap_long_path):
    print("✅ Local SHAP exists -> skip:", local_shap_long_path)
else:
    explainer_local = shap.Explainer(f_shap, X_bg, feature_names=feature_names)
    # chunk for safety
    CHUNK = 100
    all_long = []
    X_local_np = X_local.values  # keep order
    for start in range(0, len(X_local_np), CHUNK):
        end = min(start + CHUNK, len(X_local_np))
        sv = explainer_local(X_local.iloc[start:end], max_evals=max_evals)
        vals = sv.values
        if vals.ndim != 2:
            vals = np.array(vals).reshape(vals.shape[0], -1)

        chunk_idx = local_idx[start:end]
        all_long.append(local_shap_to_long(vals, chunk_idx, feature_names))
        print(f"  chunk {start}:{end} done")

    df_long = pd.concat(all_long, ignore_index=True)
    df_long.to_csv(local_shap_long_path, index=False)

    meta = {
        "ensemble": ENS_NAME,
        "seed": SEED,
        "n_local": int(len(local_idx)),
        "background_n": int(len(bg_idx)),
        "master1500_file": MASTER1500_PATH,
        "bg200_file": BG200_PATH,
        "max_evals": max_evals
    }
    save_json(local_shap_meta_path, meta)

    print("✅ Local SHAP saved:", local_shap_long_path)

# ============================================================
# (4) LIME on same local samples (resume-safe)
# ============================================================
LIME_DIR = os.path.join(LOCAL_DIR, "lime")
ensure_dir(LIME_DIR)

lime_long_path = os.path.join(LIME_DIR, "lime_explanations_long.csv")
lime_meta_path = os.path.join(LIME_DIR, "lime_meta.json")

print("\n=== (4) LIME (same samples) ===")
if os.path.exists(lime_long_path):
    print("✅ LIME exists -> skip:", lime_long_path)
else:
    # LIME expects numpy arrays; we explain on RAW feature space (same columns)
    # Predict function for LIME must return (n,2) probabilities
    def lime_predict_fn(X_np):
        Xdf = pd.DataFrame(X_np, columns=feature_names)
        return hard_ensemble_predict_proba(Xdf)

    # Build explainer on training data (RAW features)
    explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=feature_names,
        class_names=["benign(0)", "phishing(1)"],
        discretize_continuous=True,
        mode="classification",
        random_state=SEED
    )

    rows = []
    for i, ridx in enumerate(local_idx):
        x_np = X.iloc[ridx].values
        exp = explainer.explain_instance(
            data_row=x_np,
            predict_fn=lime_predict_fn,
            num_features=LOCAL_TOP_FEATURES,
        )

        # exp.as_list() gives list of (feature_rule, weight) like "hostname_length <= 23.5"
        # We'll store exactly what LIME returns; comparisons are still valid.
        for rank, (feat_rule, weight) in enumerate(exp.as_list(), start=1):
            rows.append({
                "row_idx": int(ridx),
                "rank": int(rank),
                "feature_rule": str(feat_rule),
                "weight": float(weight),
                "abs_weight": float(abs(weight)),
                "sign": 1 if weight >= 0 else -1
            })

        if (i+1) % 50 == 0:
            print(f"  explained {i+1}/{len(local_idx)}")

    lime_long = pd.DataFrame(rows)
    lime_long.to_csv(lime_long_path, index=False)

    meta = {
        "ensemble": ENS_NAME,
        "seed": SEED,
        "n_local": int(len(local_idx)),
        "num_features": LOCAL_TOP_FEATURES,
        "discretize_continuous": True,
        "note": "feature_rule is LIME's textual bin/rule; kept as-is for faithful comparison."
    }
    save_json(lime_meta_path, meta)

    print("✅ LIME saved:", lime_long_path)

print("\n========================")
print("✅ EnsembleHard DONE")
print("Outputs:")
print(" -", ENS_DIR)
print("========================")

✅ Feature count: 75 | max_evals: 151
✅ Base models: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
✅ MASTER1500 n = 1500 | BG200 n = 200
✅ Test pool n = 115984 | Train pool n = 463936

=== (1) Global SHAP fixed-step ===


PermutationExplainer explainer: 201it [03:40,  1.15s/it]                                                               


[Global SHAP] ✅ computed -> N=200


PermutationExplainer explainer: 501it [09:21,  1.14s/it]                                                               


[Global SHAP] ✅ computed -> N=500


...

  chunk 100:200 done


PermutationExplainer explainer: 101it [01:47,  1.18s/it]                                                               


  chunk 200:300 done


PermutationExplainer explainer: 101it [01:47,  1.19s/it]                                                               


  chunk 300:400 done
✅ Local SHAP saved: .\outputs\ENSEMBLES\EnsembleHard\local_xai_400\local_shap\shap_local_long.csv

=== (4) LIME (same samples) ===
  explained 50/400
  explained 100/400
  explained 150/400
  explained 200/400
  explained 250/400
  explained 300/400
  explained 350/400
  explained 400/400
✅ LIME saved: .\outputs\ENSEMBLES\EnsembleHard\local_xai_400\lime\lime_explanations_long.csv

✅ EnsembleHard DONE
Outputs:
 - .\outputs\ENSEMBLES\EnsembleHard


In [36]:
# ============================================================
# ENSEMBLE STACKING (Meta: LogisticRegression)
# Global SHAP fixed-step + Local SHAP + LIME
# Resume-safe: if output files exist -> skip
# Uses SAME master1500 (Soft file) + SAME bg200 (shared file)
# Reuses SAME local samples 400 if possible (prefer EnsembleHard -> CatBoost -> else create)
# ============================================================

import os, glob, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import shap
import joblib

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression

from lime.lime_tabular import LimeTabularExplainer

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
KFOLDS = 5
STEPS = [200, 500, 1000, 1500]
TOPK = 30
LOCAL_N_TOTAL = 400
LOCAL_TOP_FEATURES = 10

OUT_ROOT = r".\outputs"
ENS_NAME = "EnsembleStacking"
ENS_DIR  = os.path.join(OUT_ROOT, "ENSEMBLES", ENS_NAME)

MASTER1500_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_step_master_testset_1500.csv"
BG200_PATH      = r".\outputs\_shared\background_200_idx.csv"
TEST_IDX_PATH   = r".\outputs\_shared\split_indices_test.csv"
TRAIN_IDX_PATH  = r".\outputs\_shared\split_indices_train.csv"

os.makedirs(ENS_DIR, exist_ok=True)

# ----------------------------
# Preconditions
# ----------------------------
if "pipelines" not in globals() or not isinstance(pipelines, dict) or len(pipelines) == 0:
    raise RuntimeError("pipelines dict bulunamadı. Aşama2’de fitted pipelines olmalı.")

# X,y yoksa df'den türet
if "X" not in globals() or "y" not in globals():
    if "df" in globals() and "label" in df.columns:
        y = df["label"].astype(int)
        drop_cols = [c for c in ["label", "url"] if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
    else:
        raise RuntimeError("X/y bulunamadı. Aşama2’de X,y veya df(label) olmalı.")

feature_names = list(X.columns)
d = len(feature_names)
max_evals = int(2*d + 1)

base_model_names = list(pipelines.keys())
m = len(base_model_names)

print("✅ Feature count:", d, "| max_evals:", max_evals)
print("✅ Base models:", base_model_names)

# ----------------------------
# Helpers
# ----------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_json(path, obj):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_idx_csv(path):
    df = pd.read_csv(path)
    return df[df.columns[0]].to_numpy()

def find_first(patterns, base_dir=OUT_ROOT):
    candidates = []
    for pat in patterns:
        candidates += glob.glob(os.path.join(base_dir, "**", pat), recursive=True)
    candidates = sorted(set(candidates))
    return candidates[0] if candidates else None

# ----------------------------
# Load indices (NO re-split)
# ----------------------------
master_idx = read_idx_csv(MASTER1500_PATH)  # 'test_index'
bg_idx     = read_idx_csv(BG200_PATH)       # 'bg_idx'
test_idx   = read_idx_csv(TEST_IDX_PATH)    # 'test_idx'
train_idx  = read_idx_csv(TRAIN_IDX_PATH)   # 'train_idx'

X_master = X.iloc[master_idx].copy()
X_bg     = X.iloc[bg_idx].copy()

X_test   = X.iloc[test_idx].copy()
y_test   = y.iloc[test_idx].copy()

X_train  = X.iloc[train_idx].copy()
y_train  = y.iloc[train_idx].copy()

print("✅ MASTER1500 n =", len(master_idx), "| BG200 n =", len(bg_idx))
print("✅ Test pool n =", len(test_idx), "| Train pool n =", len(train_idx))

# ============================================================
# (0) Train/Load meta-model (resume-safe)
# Stacking uses OOF predictions on TRAIN to fit LogisticRegression meta-model.
# Base models for inference are the already-fitted pipelines in `pipelines`.
# ============================================================
STACK_DIR = os.path.join(ENS_DIR, "stacking_model")
ensure_dir(STACK_DIR)

meta_model_path = os.path.join(STACK_DIR, "meta_model_lr.joblib")
oof_path = os.path.join(STACK_DIR, "oof_meta_features_train.npz")
stack_meta_path = os.path.join(STACK_DIR, "stacking_training_meta.json")

if os.path.exists(meta_model_path):
    meta_lr = joblib.load(meta_model_path)
    print("\n✅ Meta-model exists -> loaded:", meta_model_path)
else:
    print("\n=== Training meta-model via OOF predictions (resume-safe) ===")

    # Create OOF meta-features: shape (n_train, m)
    n_train = len(X_train)
    oof = np.zeros((n_train, m), dtype=float)

    skf = StratifiedKFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)

    X_train_np = X_train  # keep df for pipelines
    y_train_np = y_train.values

    for fold, (tr_idx_local, va_idx_local) in enumerate(skf.split(np.zeros(n_train), y_train_np), start=1):
        X_tr = X_train_np.iloc[tr_idx_local]
        y_tr = y_train.iloc[tr_idx_local]
        X_va = X_train_np.iloc[va_idx_local]

        for j, name in enumerate(base_model_names):
            # clone -> fit on fold-train -> predict fold-val
            mdl = clone(pipelines[name])
            mdl.fit(X_tr, y_tr)
            oof[va_idx_local, j] = mdl.predict_proba(X_va)[:, 1]

        print(f"  fold {fold}/{KFOLDS} done")

    # Fit meta LR on OOF
    meta_lr = LogisticRegression(max_iter=2000, random_state=SEED)
    meta_lr.fit(oof, y_train.values)

    joblib.dump(meta_lr, meta_model_path)
    np.savez_compressed(oof_path, oof=oof, y=y_train.values, base_models=np.array(base_model_names, dtype=object))

    meta_info = {
        "seed": SEED,
        "kfolds": KFOLDS,
        "meta_model": "LogisticRegression(max_iter=2000)",
        "base_models": base_model_names,
        "train_rows": int(n_train),
        "note": "OOF probs used to train meta-model to reduce leakage. Base pipelines for inference are full fitted pipelines already in memory."
    }
    save_json(stack_meta_path, meta_info)

    print("✅ Meta-model trained & saved:", meta_model_path)

# Stacking predict_proba
def stacking_proba_from_df(X_df: pd.DataFrame) -> np.ndarray:
    # meta-features = base model probabilities
    Z = np.zeros((len(X_df), m), dtype=float)
    for j, name in enumerate(base_model_names):
        Z[:, j] = pipelines[name].predict_proba(X_df)[:, 1]
    # meta proba
    return meta_lr.predict_proba(Z)[:, 1]

def stacking_predict_proba(X_df: pd.DataFrame) -> np.ndarray:
    p1 = stacking_proba_from_df(X_df)
    p0 = 1.0 - p1
    return np.column_stack([p0, p1])

# SHAP function f(X)->p1
def f_shap(Xinput):
    if isinstance(Xinput, np.ndarray):
        Xdf = pd.DataFrame(Xinput, columns=feature_names)
    else:
        Xdf = Xinput
    return stacking_proba_from_df(Xdf)

# Save ensemble meta once
ens_meta_path = os.path.join(ENS_DIR, "meta.json")
if not os.path.exists(ens_meta_path):
    meta = {
        "ensemble_type": "Stacking",
        "meta_model": "LogisticRegression",
        "seed": SEED,
        "kfolds": KFOLDS,
        "base_models": base_model_names,
        "master1500_file": MASTER1500_PATH,
        "bg200_file": BG200_PATH,
        "fixed_steps": STEPS,
        "max_evals": max_evals,
        "feature_count": d,
        "local_n_total": LOCAL_N_TOTAL,
        "local_top_features": LOCAL_TOP_FEATURES
    }
    save_json(ens_meta_path, meta)

# ============================================================
# (1) GLOBAL SHAP fixed-step (resume-safe)
# ============================================================
GLOBAL_DIR = os.path.join(ENS_DIR, "global_shap_fixedstep_1500_200")
ensure_dir(GLOBAL_DIR)

def compute_global_step(n):
    step_csv = os.path.join(GLOBAL_DIR, f"shap_importance_N{n}.csv")
    step_npz = os.path.join(GLOBAL_DIR, f"shap_values_N{n}.npz")

    if os.path.exists(step_csv):
        print(f"[Global SHAP] ✅ exists -> N={n} (skip)")
        return

    X_step = X_master.iloc[:n].copy()
    explainer = shap.Explainer(f_shap, X_bg, feature_names=feature_names)

    sv = explainer(X_step, max_evals=max_evals)
    values = sv.values
    if values.ndim != 2:
        values = np.array(values).reshape(values.shape[0], -1)

    imp = np.mean(np.abs(values), axis=0)
    imp_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": imp})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp_df["rank"] = np.arange(1, len(imp_df) + 1)

    imp_df.to_csv(step_csv, index=False)
    np.savez_compressed(step_npz, shap_values=values)
    print(f"[Global SHAP] ✅ computed -> N={n}")

print("\n=== (1) Global SHAP fixed-step (Stacking) ===")
for n in STEPS:
    compute_global_step(n)

# merge table
dfs = []
for n in STEPS:
    df_n = pd.read_csv(os.path.join(GLOBAL_DIR, f"shap_importance_N{n}.csv"))[["feature", "mean_abs_shap"]]
    df_n = df_n.rename(columns={"mean_abs_shap": f"N{n}"})
    dfs.append(df_n)
merged = dfs[0]
for df_n in dfs[1:]:
    merged = merged.merge(df_n, on="feature", how="left")
merged.to_csv(os.path.join(GLOBAL_DIR, "shap_importance_steps_merged.csv"), index=False)

top1500 = pd.read_csv(os.path.join(GLOBAL_DIR, "shap_importance_N1500.csv"))
top1500.head(TOPK).to_csv(os.path.join(GLOBAL_DIR, f"top{TOPK}_features_N1500.csv"), index=False)

# ============================================================
# (2) LOCAL SAMPLES: reuse existing (prefer EnsembleHard) (resume-safe)
# ============================================================
LOCAL_DIR = os.path.join(ENS_DIR, "local_xai_400")
ensure_dir(LOCAL_DIR)

lime_samples_path = os.path.join(LOCAL_DIR, "lime_samples_400.csv")
lime_samples_meta = os.path.join(LOCAL_DIR, "lime_samples_meta.json")

# Prefer using the exact sample file already used for EnsembleHard (best for comparison)
preferred_same_samples = r".\outputs\ENSEMBLES\EnsembleHard\local_xai_400\lime_samples_400.csv"
existing_samples = preferred_same_samples if os.path.exists(preferred_same_samples) else find_first(
    ["*lime_samples*.csv", "*local_samples*.csv"], base_dir=OUT_ROOT
)

def create_local_samples_400():
    # Build 200 correct + 200 wrong (if possible) based on STACKING predictions on test set
    p_test = stacking_proba_from_df(X_test)
    y_pred = (p_test >= 0.5).astype(int)
    y_true = y_test.values

    correct_mask = (y_pred == y_true)
    wrong_mask = ~correct_mask

    correct_pool = test_idx[correct_mask]
    wrong_pool   = test_idx[wrong_mask]

    rng = np.random.RandomState(SEED)
    n_each = LOCAL_N_TOTAL // 2

    if len(correct_pool) >= n_each:
        rng.shuffle(correct_pool)
        chosen_correct = correct_pool[:n_each]
    else:
        chosen_correct = correct_pool

    if len(wrong_pool) >= n_each:
        rng.shuffle(wrong_pool)
        chosen_wrong = wrong_pool[:n_each]
    else:
        chosen_wrong = wrong_pool

    chosen = np.concatenate([chosen_correct, chosen_wrong])
    rng.shuffle(chosen)

    out = pd.DataFrame({"row_idx": chosen})
    out.to_csv(lime_samples_path, index=False)

    meta = {
        "created_by": ENS_NAME,
        "seed": SEED,
        "strategy": "200 correct + 200 wrong (if available) based on stacking predictions",
        "n_selected": int(len(out))
    }
    save_json(lime_samples_meta, meta)

    print("✅ Local samples CREATED:", lime_samples_path, "| n=", len(out))
    return out

# Reuse / create
if os.path.exists(lime_samples_path):
    local_samples_df = pd.read_csv(lime_samples_path)
    print("\n✅ Reusing local samples (already in EnsembleStacking folder):", lime_samples_path)
else:
    if existing_samples is not None:
        tmp = pd.read_csv(existing_samples)
        # Accept common column names
        if "row_idx" in tmp.columns:
            col = "row_idx"
        elif "idx" in tmp.columns:
            col = "idx"
        elif "test_index" in tmp.columns:
            col = "test_index"
        elif "test_idx" in tmp.columns:
            col = "test_idx"
        elif "sample_id" in tmp.columns:
            col = "sample_id"
        else:
            col = tmp.columns[0]

        vals = tmp[col].to_numpy()
        vals = vals[:LOCAL_N_TOTAL] if len(vals) >= LOCAL_N_TOTAL else vals

        local_samples_df = pd.DataFrame({"row_idx": vals})
        local_samples_df.to_csv(lime_samples_path, index=False)

        meta = {
            "reused_from": existing_samples,
            "used_column": col,
            "n_source": int(len(tmp)),
            "n_used": int(len(local_samples_df)),
            "note": "Reused previous sample set to enable direct comparisons."
        }
        save_json(lime_samples_meta, meta)

        print("\n✅ Reused previous sample file:", existing_samples)
        print("   Saved as:", lime_samples_path, "| n=", len(local_samples_df))
    else:
        print("\n⚠️ No previous samples found. Creating a new local sample set.")
        local_samples_df = create_local_samples_400()

local_idx = local_samples_df["row_idx"].to_numpy()
X_local = X.iloc[local_idx].copy()
y_local = y.iloc[local_idx].copy()

# ============================================================
# (3) LOCAL SHAP on same local samples (resume-safe)
# ============================================================
LOCAL_SHAP_DIR = os.path.join(LOCAL_DIR, "local_shap")
ensure_dir(LOCAL_SHAP_DIR)

local_shap_long_path = os.path.join(LOCAL_SHAP_DIR, "shap_local_long.csv")
local_shap_meta_path = os.path.join(LOCAL_SHAP_DIR, "shap_local_meta.json")

def local_shap_to_long(shap_values, row_indices, feature_names):
    abs_vals = np.abs(shap_values)
    ranks = (-abs_vals).argsort(axis=1)
    rows = []
    for i in range(shap_values.shape[0]):
        idx_ = row_indices[i]
        rank_pos = np.empty(shap_values.shape[1], dtype=int)
        rank_pos[ranks[i]] = np.arange(1, shap_values.shape[1] + 1)
        for j, fn in enumerate(feature_names):
            v = shap_values[i, j]
            rows.append((idx_, fn, float(v), float(abs(v)), int(rank_pos[j]), 1 if v >= 0 else -1))
    return pd.DataFrame(rows, columns=["row_idx", "feature", "shap_value", "abs_shap", "rank_abs", "sign"])

print("\n=== (3) Local SHAP (Stacking) ===")
if os.path.exists(local_shap_long_path):
    print("✅ Local SHAP exists -> skip:", local_shap_long_path)
else:
    explainer_local = shap.Explainer(f_shap, X_bg, feature_names=feature_names)
    CHUNK = 100
    all_long = []
    for start in range(0, len(local_idx), CHUNK):
        end = min(start + CHUNK, len(local_idx))
        sv = explainer_local(X_local.iloc[start:end], max_evals=max_evals)
        vals = sv.values
        if vals.ndim != 2:
            vals = np.array(vals).reshape(vals.shape[0], -1)
        chunk_idx = local_idx[start:end]
        all_long.append(local_shap_to_long(vals, chunk_idx, feature_names))
        print(f"  chunk {start}:{end} done")
    df_long = pd.concat(all_long, ignore_index=True)
    df_long.to_csv(local_shap_long_path, index=False)

    meta = {
        "ensemble": ENS_NAME,
        "seed": SEED,
        "n_local": int(len(local_idx)),
        "background_n": int(len(bg_idx)),
        "max_evals": max_evals,
        "bg200_file": BG200_PATH
    }
    save_json(local_shap_meta_path, meta)

    print("✅ Local SHAP saved:", local_shap_long_path)

# ============================================================
# (4) LIME on same local samples (resume-safe)
# ============================================================
LIME_DIR = os.path.join(LOCAL_DIR, "lime")
ensure_dir(LIME_DIR)

lime_long_path = os.path.join(LIME_DIR, "lime_explanations_long.csv")
lime_meta_path = os.path.join(LIME_DIR, "lime_meta.json")

print("\n=== (4) LIME (Stacking) ===")
if os.path.exists(lime_long_path):
    print("✅ LIME exists -> skip:", lime_long_path)
else:
    def lime_predict_fn(X_np):
        Xdf = pd.DataFrame(X_np, columns=feature_names)
        return stacking_predict_proba(Xdf)

    explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=feature_names,
        class_names=["benign(0)", "phishing(1)"],
        discretize_continuous=True,
        mode="classification",
        random_state=SEED
    )

    rows = []
    for i, ridx in enumerate(local_idx):
        x_np = X.iloc[ridx].values
        exp = explainer.explain_instance(
            data_row=x_np,
            predict_fn=lime_predict_fn,
            num_features=LOCAL_TOP_FEATURES,
        )
        for rank, (feat_rule, weight) in enumerate(exp.as_list(), start=1):
            rows.append({
                "row_idx": int(ridx),
                "rank": int(rank),
                "feature_rule": str(feat_rule),
                "weight": float(weight),
                "abs_weight": float(abs(weight)),
                "sign": 1 if weight >= 0 else -1
            })
        if (i+1) % 50 == 0:
            print(f"  explained {i+1}/{len(local_idx)}")

    lime_long = pd.DataFrame(rows)
    lime_long.to_csv(lime_long_path, index=False)

    meta = {
        "ensemble": ENS_NAME,
        "seed": SEED,
        "n_local": int(len(local_idx)),
        "num_features": LOCAL_TOP_FEATURES,
        "discretize_continuous": True,
        "note": "feature_rule is LIME's textual bin/rule; kept as-is for faithful comparison."
    }
    save_json(lime_meta_path, meta)

    print("✅ LIME saved:", lime_long_path)

print("\n========================")
print("✅ EnsembleStacking DONE")
print("Outputs:")
print(" -", ENS_DIR)
print("========================")

✅ Feature count: 75 | max_evals: 151
✅ Base models: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
✅ MASTER1500 n = 1500 | BG200 n = 200
✅ Test pool n = 115984 | Train pool n = 463936

=== Training meta-model via OOF predictions (resume-safe) ===
[LightGBM] [Info] Number of positive: 154141, number of negative: 217007
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.120121 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6140
[LightGBM] [Info] Number of data points in the train set: 371148, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342062
[LightGBM] [Info] Start training from score -0.342062
  fold 1/5 done
[LightGBM] [Info] Number of positive: 154142, number of negative: 217007
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead o

PermutationExplainer explainer: 201it [03:42,  1.17s/it]                                                               


[Global SHAP] ✅ computed -> N=200


PermutationExplainer explainer: 501it [09:34,  1.17s/it]                                                               


[Global SHAP] ✅ computed -> N=500


...

  chunk 100:200 done


PermutationExplainer explainer: 101it [01:56,  1.28s/it]                                                               


  chunk 200:300 done


PermutationExplainer explainer: 101it [02:01,  1.32s/it]                                                               


  chunk 300:400 done
✅ Local SHAP saved: .\outputs\ENSEMBLES\EnsembleStacking\local_xai_400\local_shap\shap_local_long.csv

=== (4) LIME (Stacking) ===
  explained 50/400
  explained 100/400
  explained 150/400
  explained 200/400
  explained 250/400
  explained 300/400
  explained 350/400
  explained 400/400
✅ LIME saved: .\outputs\ENSEMBLES\EnsembleStacking\local_xai_400\lime\lime_explanations_long.csv

✅ EnsembleStacking DONE
Outputs:
 - .\outputs\ENSEMBLES\EnsembleStacking


In [37]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_ROOT = r".\outputs"
FIG_DIR = os.path.join(OUT_ROOT, "_shared", "paper_figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Models we want in comparison
TARGETS = [
    ("EnsembleSoft",   os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleSoft")),
    ("EnsembleHard",   os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleHard")),
    ("EnsembleStacking", os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleStacking")),
    ("LogReg", os.path.join(OUT_ROOT, "LogReg")),
    ("DecisionTree", os.path.join(OUT_ROOT, "DecisionTree")),
    ("RandomForest", os.path.join(OUT_ROOT, "RandomForest")),
    ("XGBoost", os.path.join(OUT_ROOT, "XGBoost")),
    ("LightGBM", os.path.join(OUT_ROOT, "LightGBM")),
    ("CatBoost", os.path.join(OUT_ROOT, "CatBoost")),
]

TOPK = 20  # use Top-20 for overlap & heatmaps

def find_shap1500_csv(base_dir):
    # search in known folder structure
    pats = [
        "**/global_shap_fixedstep_1500_200/shap_importance_N1500.csv",
        "**/global_shap_fixedstep_1500_200/shap_importance_N1500.csv",
        "**/global/**/shap_importance.csv",
        "**/global/shap_importance.csv",
    ]
    cands = []
    for p in pats:
        cands += glob.glob(os.path.join(base_dir, p), recursive=True)
    cands = sorted(set(cands))
    return cands[0] if cands else None

def load_importance(path):
    df = pd.read_csv(path)
    # normalize column naming
    if "mean_abs_shap" not in df.columns:
        # some older files might use different names
        for c in df.columns:
            if "shap" in c.lower() and "abs" in c.lower():
                df = df.rename(columns={c: "mean_abs_shap"})
                break
    df = df[["feature", "mean_abs_shap"]].copy()
    df = df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    return df

# 1) Load all
loaded = {}
paths = {}
for name, base in TARGETS:
    p = find_shap1500_csv(base)
    if p is None:
        print(f"⚠️ Not found shap N1500 for {name} under {base}")
        continue
    paths[name] = p
    loaded[name] = load_importance(p)
    print(f"✅ Loaded {name}: {p}")

if len(loaded) < 3:
    raise RuntimeError("Too few models loaded. Check your Soft/Hard/Stacking paths.")

# 2) Build union feature set (TopK union for heatmap)
top_union = set()
for name, df in loaded.items():
    top_union |= set(df.head(TOPK)["feature"].tolist())
top_union = sorted(top_union)

# Matrix: model × feature (raw importance)
model_names = list(loaded.keys())
raw_mat = pd.DataFrame(index=model_names, columns=top_union, dtype=float)

for name in model_names:
    df = loaded[name].set_index("feature")
    for f in top_union:
        raw_mat.loc[name, f] = float(df.loc[f, "mean_abs_shap"]) if f in df.index else 0.0

raw_mat_path = os.path.join(FIG_DIR, f"global_shap_raw_matrix_top{TOPK}.csv")
raw_mat.to_csv(raw_mat_path)

# Row-normalized matrix
row_norm = raw_mat.div(raw_mat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
row_norm_path = os.path.join(FIG_DIR, f"global_shap_row_norm_matrix_top{TOPK}.csv")
row_norm.to_csv(row_norm_path)

# 3) Overlap / Jaccard matrix on TopK
top_sets = {name: set(loaded[name].head(TOPK)["feature"]) for name in model_names}
jac = pd.DataFrame(index=model_names, columns=model_names, dtype=float)

for a in model_names:
    for b in model_names:
        inter = len(top_sets[a] & top_sets[b])
        union = len(top_sets[a] | top_sets[b])
        jac.loc[a, b] = inter / union if union else 0.0

jac_path = os.path.join(FIG_DIR, f"top{TOPK}_jaccard_overlap_matrix.csv")
jac.to_csv(jac_path)

# 4) Rank correlation (Spearman) using union features (rank by importance)
# Build ranks per model on union features
ranks = {}
for name in model_names:
    df = loaded[name].copy()
    # rank all features present in df; missing gets worst rank
    feat_rank = {f: i+1 for i, f in enumerate(df["feature"].tolist())}
    worst = len(feat_rank) + 1
    ranks[name] = np.array([feat_rank.get(f, worst) for f in top_union])

spearman = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for i, a in enumerate(model_names):
    ra = ranks[a]
    for j, b in enumerate(model_names):
        rb = ranks[b]
        # spearman corr = corr(rank vectors)
        if np.std(ra) == 0 or np.std(rb) == 0:
            spearman.loc[a, b] = 0.0
        else:
            spearman.loc[a, b] = float(np.corrcoef(ra, rb)[0, 1])

spearman_path = os.path.join(FIG_DIR, f"top{TOPK}_rankcorr_matrix.csv")
spearman.to_csv(spearman_path)

# 5) Save heatmaps (raw and row-normalized)
def save_heatmap(df, out_png, title):
    # simple matplotlib imshow (no special colors specified)
    arr = df.values.astype(float)
    plt.figure(figsize=(max(8, len(df.columns)*0.35), max(4, len(df.index)*0.45)))
    plt.imshow(arr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(df.columns)), df.columns, rotation=90)
    plt.yticks(range(len(df.index)), df.index)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

save_heatmap(raw_mat, os.path.join(FIG_DIR, f"heatmap_raw_top{TOPK}.png"),
             f"Global SHAP mean(|SHAP|) heatmap (Top-{TOPK} union, raw)")

save_heatmap(row_norm, os.path.join(FIG_DIR, f"heatmap_row_norm_top{TOPK}.png"),
             f"Global SHAP heatmap (Top-{TOPK} union, row-normalized)")

save_heatmap(jac, os.path.join(FIG_DIR, f"jaccard_overlap_top{TOPK}.png"),
             f"Top-{TOPK} Jaccard overlap (features)")

save_heatmap(spearman, os.path.join(FIG_DIR, f"rankcorr_top{TOPK}.png"),
             f"Rank correlation (proxy Spearman) on Top-{TOPK} union")

# 6) Delta importance bars for ensemble pairs (Top features by absolute delta)
def delta_bar(a, b, k=20):
    # delta = a - b
    da = loaded[a].set_index("feature")["mean_abs_shap"]
    db = loaded[b].set_index("feature")["mean_abs_shap"]
    feats = sorted(set(da.index) | set(db.index))
    delta = pd.Series({f: float(da.get(f, 0.0) - db.get(f, 0.0)) for f in feats})
    top = delta.abs().sort_values(ascending=False).head(k)
    out_csv = os.path.join(FIG_DIR, f"delta_{a}_minus_{b}_top{k}.csv")
    top.to_csv(out_csv, header=["delta_abs"])
    # plot
    plt.figure(figsize=(10, max(4, k*0.25)))
    plt.barh(top.index[::-1], delta.loc[top.index][::-1])
    plt.title(f"Delta importance: {a} − {b} (Top-{k} |Δ| features)")
    plt.tight_layout()
    out_png = os.path.join(FIG_DIR, f"delta_{a}_minus_{b}_top{k}.png")
    plt.savefig(out_png, dpi=200)
    plt.close()

# If all exist, create deltas
pairs = [("EnsembleHard","EnsembleSoft"), ("EnsembleStacking","EnsembleSoft"), ("EnsembleStacking","EnsembleHard")]
for a,b in pairs:
    if a in loaded and b in loaded:
        delta_bar(a,b,k=20)

# 7) Save a manifest for paper writing
manifest = {
    "loaded_shap1500_paths": paths,
    "outputs": {
        "raw_matrix_csv": raw_mat_path,
        "row_norm_matrix_csv": row_norm_path,
        "jaccard_csv": jac_path,
        "rankcorr_csv": spearman_path,
        "fig_dir": FIG_DIR
    }
}
with open(os.path.join(FIG_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n✅ Paper figures generated in:", FIG_DIR)
print("Key files:")
print(" -", raw_mat_path)
print(" -", row_norm_path)
print(" -", jac_path)
print(" -", spearman_path)
print(" - heatmap_raw_top20.png / heatmap_row_norm_top20.png / jaccard_overlap_top20.png / rankcorr_top20.png")

⚠️ Not found shap N1500 for EnsembleSoft under .\outputs\ENSEMBLES\EnsembleSoft
✅ Loaded EnsembleHard: .\outputs\ENSEMBLES\EnsembleHard\global_shap_fixedstep_1500_200\shap_importance_N1500.csv
✅ Loaded EnsembleStacking: .\outputs\ENSEMBLES\EnsembleStacking\global_shap_fixedstep_1500_200\shap_importance_N1500.csv
✅ Loaded LogReg: .\outputs\LogReg\global\shap_importance.csv
✅ Loaded DecisionTree: .\outputs\DecisionTree\global\shap_importance.csv
✅ Loaded RandomForest: .\outputs\RandomForest\global\shap_importance.csv
✅ Loaded XGBoost: .\outputs\XGBoost\global\shap_importance.csv
✅ Loaded LightGBM: .\outputs\LightGBM\global\shap_importance.csv
✅ Loaded CatBoost: .\outputs\CatBoost\global\shap_importance.csv

✅ Paper figures generated in: .\outputs\_shared\paper_figures
Key files:
 - .\outputs\_shared\paper_figures\global_shap_raw_matrix_top20.csv
 - .\outputs\_shared\paper_figures\global_shap_row_norm_matrix_top20.csv
 - .\outputs\_shared\paper_figures\top20_jaccard_overlap_matrix.csv
 - 

In [38]:
import os
import glob
import numpy as np
import pandas as pd

OUT_ROOT = r".\outputs"
SAVE_DIR = os.path.join(OUT_ROOT, "_shared", "final_metrics")
os.makedirs(SAVE_DIR, exist_ok=True)

TOPK = 20

MODELS = [
    "EnsembleSoft",
    "EnsembleHard",
    "EnsembleStacking",
    "LogReg",
    "DecisionTree",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost"
]

def find_shap_1500(model):
    if model.startswith("Ensemble"):
        base = os.path.join(OUT_ROOT, "ENSEMBLES", model)
    else:
        base = os.path.join(OUT_ROOT, model)
    paths = glob.glob(os.path.join(base, "**", "shap_importance_N1500.csv"), recursive=True)
    return paths[0] if paths else None

def load_importance(path):
    df = pd.read_csv(path)
    df = df[["feature", "mean_abs_shap"]]
    df = df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    return df

# Load all
data = {}
paths = {}

for m in MODELS:
    p = find_shap_1500(m)
    if p:
        data[m] = load_importance(p)
        paths[m] = p
        print(f"Loaded {m}")
    else:
        print(f"Not found: {m}")

model_names = list(data.keys())

# ---------- RAW MATRIX ----------
all_features = set()
for m in model_names:
    all_features |= set(data[m].head(TOPK)["feature"])
all_features = sorted(all_features)

raw_mat = pd.DataFrame(index=model_names, columns=all_features)

for m in model_names:
    df = data[m].set_index("feature")
    for f in all_features:
        raw_mat.loc[m, f] = df.loc[f, "mean_abs_shap"] if f in df.index else 0

raw_mat = raw_mat.astype(float)
raw_mat.to_csv(os.path.join(SAVE_DIR, "raw_importance_matrix_top20.csv"))

# ---------- ROW NORMALIZED ----------
row_norm = raw_mat.div(raw_mat.sum(axis=1), axis=0)
row_norm.to_csv(os.path.join(SAVE_DIR, "row_norm_matrix_top20.csv"))

# ---------- JACCARD ----------
top_sets = {m: set(data[m].head(TOPK)["feature"]) for m in model_names}

jac = pd.DataFrame(index=model_names, columns=model_names)

for a in model_names:
    for b in model_names:
        inter = len(top_sets[a] & top_sets[b])
        union = len(top_sets[a] | top_sets[b])
        jac.loc[a, b] = inter / union

jac = jac.astype(float)
jac.to_csv(os.path.join(SAVE_DIR, "jaccard_top20.csv"))

# ---------- RANK CORRELATION ----------
rankcorr = pd.DataFrame(index=model_names, columns=model_names)

for a in model_names:
    rank_a = {f: i for i, f in enumerate(data[a]["feature"])}
    for b in model_names:
        rank_b = {f: i for i, f in enumerate(data[b]["feature"])}
        common = list(set(rank_a.keys()) & set(rank_b.keys()))
        ra = [rank_a[f] for f in common]
        rb = [rank_b[f] for f in common]
        if len(ra) > 2:
            corr = np.corrcoef(ra, rb)[0,1]
        else:
            corr = 0
        rankcorr.loc[a, b] = corr

rankcorr = rankcorr.astype(float)
rankcorr.to_csv(os.path.join(SAVE_DIR, "rankcorr_top20.csv"))

# ---------- DELTA TABLES ----------
def delta(a, b):
    df_a = data[a].set_index("feature")["mean_abs_shap"]
    df_b = data[b].set_index("feature")["mean_abs_shap"]
    feats = sorted(set(df_a.index) | set(df_b.index))
    delta = pd.Series({f: df_a.get(f,0) - df_b.get(f,0) for f in feats})
    delta = delta.reindex(delta.abs().sort_values(ascending=False).index)
    return delta

pairs = [
    ("EnsembleHard","EnsembleSoft"),
    ("EnsembleStacking","EnsembleSoft"),
    ("EnsembleStacking","EnsembleHard")
]

for a,b in pairs:
    if a in data and b in data:
        d = delta(a,b)
        d.head(30).to_csv(os.path.join(SAVE_DIR, f"delta_{a}_minus_{b}.csv"))

print("\n✅ GLOBAL COMPARISON FILES SAVED TO:", SAVE_DIR)

Not found: EnsembleSoft
Loaded EnsembleHard
Loaded EnsembleStacking
Loaded LogReg
Loaded DecisionTree
Loaded RandomForest
Loaded XGBoost
Loaded LightGBM
Loaded CatBoost

✅ GLOBAL COMPARISON FILES SAVED TO: .\outputs\_shared\final_metrics


In [39]:
import glob, os

OUT_ROOT = r".\outputs"
soft_dir = os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleSoft")

patterns = [
    os.path.join(soft_dir, "**", "shap_importance_N1500.csv"),
    os.path.join(soft_dir, "**", "shap_importance.csv"),
    os.path.join(soft_dir, "**", "*shap*1500*.csv"),
    os.path.join(soft_dir, "**", "*shap_importance*.csv"),
]

cands = []
for p in patterns:
    cands += glob.glob(p, recursive=True)

cands = sorted(set(cands))
print("Found", len(cands), "candidates under:", soft_dir)
for i, f in enumerate(cands[:50], start=1):
    print(f"{i:02d}) {f}")

Found 8 candidates under: .\outputs\ENSEMBLES\EnsembleSoft
01) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_stability_fixed_200_500_1000_1500.csv
02) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps\N1000\shap_importance.csv
03) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps\N200\shap_importance.csv
04) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps\N500\shap_importance.csv
05) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N1000\shap_importance.csv
06) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N1500\shap_importance.csv
07) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N200\shap_importance.csv
08) .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N500\shap_importance.csv


In [40]:
import os, glob
import numpy as np
import pandas as pd

OUT_ROOT = r".\outputs"
SAVE_DIR = os.path.join(OUT_ROOT, "_shared", "final_metrics")
os.makedirs(SAVE_DIR, exist_ok=True)

TOPK = 20

# ✅ Soft'un doğru N1500 fixed-step dosyası
SOFT_SHAP_PATH = r".\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N1500\shap_importance.csv"

# diğerleri (N1500 fixed-step)
HARD_SHAP_PATH = glob.glob(os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleHard", "**", "shap_importance_N1500.csv"), recursive=True)[0]
STACK_SHAP_PATH = glob.glob(os.path.join(OUT_ROOT, "ENSEMBLES", "EnsembleStacking", "**", "shap_importance_N1500.csv"), recursive=True)[0]

SINGLE = {
    "LogReg": glob.glob(os.path.join(OUT_ROOT, "LogReg", "**", "shap_importance_N1500.csv"), recursive=True)[0],
    "DecisionTree": glob.glob(os.path.join(OUT_ROOT, "DecisionTree", "**", "shap_importance_N1500.csv"), recursive=True)[0],
    "RandomForest": glob.glob(os.path.join(OUT_ROOT, "RandomForest", "**", "shap_importance_N1500.csv"), recursive=True)[0],
    "XGBoost": glob.glob(os.path.join(OUT_ROOT, "XGBoost", "**", "shap_importance_N1500.csv"), recursive=True)[0],
    "LightGBM": glob.glob(os.path.join(OUT_ROOT, "LightGBM", "**", "shap_importance_N1500.csv"), recursive=True)[0],
    "CatBoost": glob.glob(os.path.join(OUT_ROOT, "CatBoost", "**", "shap_importance_N1500.csv"), recursive=True)[0],
}

MODELS = {
    "EnsembleSoft": SOFT_SHAP_PATH,
    "EnsembleHard": HARD_SHAP_PATH,
    "EnsembleStacking": STACK_SHAP_PATH,
    **SINGLE
}

def load_importance(path):
    df = pd.read_csv(path)
    # Soft dosyasında kolon adı farklı olabilir -> normalize et
    if "mean_abs_shap" not in df.columns:
        # shap_importance.csv bazen "mean_abs_shap" yerine "mean_abs_shap_value" vs olur
        for c in df.columns:
            if "shap" in c.lower() and "abs" in c.lower():
                df = df.rename(columns={c: "mean_abs_shap"})
                break
    # bazı dosyalarda 'feature' yerine başka isim varsa:
    if "feature" not in df.columns:
        # ilk kolonu feature gibi kabul et
        df = df.rename(columns={df.columns[0]: "feature"})
    df = df[["feature", "mean_abs_shap"]].copy()
    df = df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    return df

data = {name: load_importance(path) for name, path in MODELS.items()}

# ---------- TopK union ----------
top_union = set()
for name in data:
    top_union |= set(data[name].head(TOPK)["feature"])
top_union = sorted(top_union)

# ---------- Raw matrix ----------
raw_mat = pd.DataFrame(index=list(data.keys()), columns=top_union, dtype=float)
for name in data:
    s = data[name].set_index("feature")["mean_abs_shap"]
    raw_mat.loc[name] = [float(s.get(f, 0.0)) for f in top_union]
raw_mat.to_csv(os.path.join(SAVE_DIR, f"raw_importance_matrix_top{TOPK}.csv"))

# ---------- Row normalized ----------
row_norm = raw_mat.div(raw_mat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
row_norm.to_csv(os.path.join(SAVE_DIR, f"row_norm_matrix_top{TOPK}.csv"))

# ---------- Jaccard overlap ----------
top_sets = {name: set(data[name].head(TOPK)["feature"]) for name in data}
names = list(data.keys())

jac = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        inter = len(top_sets[a] & top_sets[b])
        union = len(top_sets[a] | top_sets[b])
        jac.loc[a, b] = inter/union if union else 0.0
jac.to_csv(os.path.join(SAVE_DIR, f"jaccard_top{TOPK}.csv"))

# ---------- Rank correlation proxy (union features; missing -> worst rank) ----------
rankcorr = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    rank_a = {f:i for i,f in enumerate(data[a]["feature"].tolist())}
    worst_a = len(rank_a) + 1
    for b in names:
        rank_b = {f:i for i,f in enumerate(data[b]["feature"].tolist())}
        worst_b = len(rank_b) + 1
        ra = np.array([rank_a.get(f, worst_a) for f in top_union], dtype=float)
        rb = np.array([rank_b.get(f, worst_b) for f in top_union], dtype=float)
        rankcorr.loc[a, b] = float(np.corrcoef(ra, rb)[0,1]) if (np.std(ra)>0 and np.std(rb)>0) else 0.0
rankcorr.to_csv(os.path.join(SAVE_DIR, f"rankcorr_top{TOPK}.csv"))

# ---------- Delta tables (Top30 by |Δ|) ----------
def delta(a,b):
    sa = data[a].set_index("feature")["mean_abs_shap"]
    sb = data[b].set_index("feature")["mean_abs_shap"]
    feats = sorted(set(sa.index)|set(sb.index))
    d = pd.Series({f: float(sa.get(f,0.0)-sb.get(f,0.0)) for f in feats})
    return d.reindex(d.abs().sort_values(ascending=False).index)

pairs=[("EnsembleHard","EnsembleSoft"), ("EnsembleStacking","EnsembleSoft"), ("EnsembleStacking","EnsembleHard")]
for a,b in pairs:
    dlt = delta(a,b).head(30)
    dlt.to_csv(os.path.join(SAVE_DIR, f"delta_{a}_minus_{b}_top30.csv"))

# Save manifest
pd.Series(MODELS).to_csv(os.path.join(SAVE_DIR, "shap1500_paths_manifest.csv"))

print("✅ Done. Saved to:", SAVE_DIR)
print("✅ Soft path:", SOFT_SHAP_PATH)

✅ Done. Saved to: .\outputs\_shared\final_metrics
✅ Soft path: .\outputs\ENSEMBLES\EnsembleSoft\global_shap_steps_fixed\N1500\shap_importance.csv


In [83]:
# ============================================================
# ENSEMBLE SOFT: Local SHAP + LIME (FORCE SAME local 400 as Hard)
# - NO TRAIN
# - Uses SAME bg200 shared file
# - Uses EXACT same local samples file from EnsembleHard
# - Overwrites Soft local outputs (so the comparison becomes valid)
# ============================================================

import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
LOCAL_TOP_FEATURES = 10

OUT_ROOT = r".\outputs"
ENS_NAME = "EnsembleSoft"
ENS_DIR  = os.path.join(OUT_ROOT, "ENSEMBLES", ENS_NAME)

BG200_PATH      = r".\outputs\_shared\background_200_idx.csv"
TRAIN_IDX_PATH  = r".\outputs\_shared\split_indices_train.csv"

# FORCE SAME samples from Hard
HARD_SAMPLES = r".\outputs\ENSEMBLES\EnsembleHard\local_xai_400\lime_samples_400.csv"

# Soft local output dirs
LOCAL_DIR = os.path.join(ENS_DIR, "local_xai_400")
LOCAL_SHAP_DIR = os.path.join(LOCAL_DIR, "local_shap")
LIME_DIR = os.path.join(LOCAL_DIR, "lime")

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(LOCAL_SHAP_DIR, exist_ok=True)
os.makedirs(LIME_DIR, exist_ok=True)

# ----------------------------
# Sanity checks
# ----------------------------
if "pipelines" not in globals() or not isinstance(pipelines, dict) or len(pipelines) == 0:
    raise RuntimeError("pipelines dict bulunamadı. Aşama2’de fitted pipelines olmalı.")

if "X" not in globals() or "y" not in globals():
    if "df" in globals() and "label" in df.columns:
        y = df["label"].astype(int)
        drop_cols = [c for c in ["label", "url"] if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
    else:
        raise RuntimeError("X/y bulunamadı. Aşama2’de X,y veya df(label) olmalı.")

if not os.path.exists(HARD_SAMPLES):
    raise RuntimeError(f"Hard sample dosyası yok: {HARD_SAMPLES}")

if not os.path.exists(BG200_PATH):
    raise RuntimeError(f"BG200 dosyası yok: {BG200_PATH}")

if not os.path.exists(TRAIN_IDX_PATH):
    raise RuntimeError(f"TRAIN idx dosyası yok: {TRAIN_IDX_PATH}")

feature_names = list(X.columns)
d = len(feature_names)
max_evals = int(2*d + 1)

print("✅ Feature count:", d, "| max_evals:", max_evals)

# ----------------------------
# Load bg200 + train indices
# ----------------------------
def read_idx_csv(path):
    df_ = pd.read_csv(path)
    return df_[df_.columns[0]].to_numpy()

bg_idx = read_idx_csv(BG200_PATH).astype(int)
train_idx = read_idx_csv(TRAIN_IDX_PATH).astype(int)

X_bg = X.iloc[bg_idx].copy()
X_train = X.iloc[train_idx].copy()

print("✅ BG200 n =", len(bg_idx), "| Train n =", len(train_idx))

# ----------------------------
# Soft predict_proba
# IMPORTANT: Eğer Aşama2’de zaten soft fonksiyonun varsa onu kullanacağız
# ----------------------------
if "ensemble_soft_predict_proba" in globals():
    def soft_predict_proba(X_df: pd.DataFrame) -> np.ndarray:
        return ensemble_soft_predict_proba(X_df)
else:
    # Fallback: average probs of base pipelines (soft voting)
    base_model_names = list(pipelines.keys())
    m = len(base_model_names)
    def soft_predict_proba(X_df: pd.DataFrame) -> np.ndarray:
        p = np.zeros(len(X_df), dtype=float)
        for name in base_model_names:
            p += pipelines[name].predict_proba(X_df)[:, 1]
        p /= float(m)
        return np.column_stack([1.0 - p, p])
    print("⚠ ensemble_soft_predict_proba yok. pipelines average fallback kullanılıyor.")

def f_shap(Xinput):
    if isinstance(Xinput, np.ndarray):
        Xdf = pd.DataFrame(Xinput, columns=feature_names)
    else:
        Xdf = Xinput
    return soft_predict_proba(Xdf)[:, 1]

# ============================================================
# (1) FORCE SAME local samples as Hard
# ============================================================
hard_samples = pd.read_csv(HARD_SAMPLES)
if "row_idx" not in hard_samples.columns:
    hard_samples = hard_samples.rename(columns={hard_samples.columns[0]: "row_idx"})

hard_samples = hard_samples.iloc[:400].copy()
hard_samples.to_csv(os.path.join(LOCAL_DIR, "lime_samples_400.csv"), index=False)

local_idx = hard_samples["row_idx"].astype(int).to_numpy()
X_local = X.iloc[local_idx].copy()

print("✅ Forced local samples from Hard. unique =", len(set(local_idx)), "/400")

# ============================================================
# (2) OVERWRITE Soft local outputs (delete old)
# ============================================================
local_shap_long_path = os.path.join(LOCAL_SHAP_DIR, "shap_local_long.csv")
lime_long_path       = os.path.join(LIME_DIR, "lime_explanations_long.csv")

for f in [local_shap_long_path, lime_long_path]:
    if os.path.exists(f):
        os.remove(f)
        print("🗑️ deleted:", f)

# ============================================================
# (3) LOCAL SHAP (Soft) -> long format with rank_abs
# ============================================================
print("\n=== Local SHAP (Soft) ===")
explainer_local = shap.Explainer(f_shap, X_bg, feature_names=feature_names)

def local_shap_to_long(shap_values, row_indices, feature_names):
    abs_vals = np.abs(shap_values)
    ranks = (-abs_vals).argsort(axis=1)
    rows = []
    for i in range(shap_values.shape[0]):
        idx_ = row_indices[i]
        rank_pos = np.empty(shap_values.shape[1], dtype=int)
        rank_pos[ranks[i]] = np.arange(1, shap_values.shape[1] + 1)
        for j, fn in enumerate(feature_names):
            v = shap_values[i, j]
            rows.append((idx_, fn, float(v), float(abs(v)), int(rank_pos[j]), 1 if v >= 0 else -1))
    return pd.DataFrame(rows, columns=["row_idx", "feature_name", "shap_value", "abs_shap", "rank_abs", "sign"])

CHUNK = 100
all_long = []
for start in range(0, len(local_idx), CHUNK):
    end = min(start + CHUNK, len(local_idx))
    sv = explainer_local(X_local.iloc[start:end], max_evals=max_evals)
    vals = sv.values
    if vals.ndim != 2:
        vals = np.array(vals).reshape(vals.shape[0], -1)
    chunk_idx = local_idx[start:end]
    all_long.append(local_shap_to_long(vals, chunk_idx, feature_names))
    print(f"  chunk {start}:{end} done")

df_long = pd.concat(all_long, ignore_index=True)
df_long.to_csv(local_shap_long_path, index=False)
print("✅ Local SHAP saved:", local_shap_long_path, "| rows:", len(df_long))

# ============================================================
# (4) LIME (Soft) -> long format with feature_rule
# ============================================================
print("\n=== LIME (Soft) ===")

def lime_predict_fn(X_np):
    Xdf = pd.DataFrame(X_np, columns=feature_names)
    return soft_predict_proba(Xdf)

explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=["benign(0)", "phishing(1)"],
    discretize_continuous=True,
    mode="classification",
    random_state=SEED
)

rows = []
for i, ridx in enumerate(local_idx):
    x_np = X.iloc[ridx].values
    exp = explainer.explain_instance(
        data_row=x_np,
        predict_fn=lime_predict_fn,
        num_features=LOCAL_TOP_FEATURES,
    )
    for rank, (feat_rule, weight) in enumerate(exp.as_list(), start=1):
        rows.append({
            "row_idx": int(ridx),
            "rank": int(rank),
            "feature_rule": str(feat_rule),
            "weight": float(weight),
            "abs_weight": float(abs(weight)),
            "sign": 1 if weight >= 0 else -1
        })
    if (i+1) % 50 == 0:
        print(f"  explained {i+1}/{len(local_idx)}")

lime_long = pd.DataFrame(rows)
lime_long.to_csv(lime_long_path, index=False)
print("✅ LIME saved:", lime_long_path, "| rows:", len(lime_long))

# ============================================================
# (5) FINAL CHECK
# ============================================================
soft_ids = set(pd.read_csv(os.path.join(LOCAL_DIR, "lime_samples_400.csv"))["row_idx"].astype(int))
hard_ids = set(pd.read_csv(HARD_SAMPLES)["row_idx"].astype(int))
print("\n✅ Intersection Hard vs Soft local samples:", len(soft_ids & hard_ids), "/400")

✅ Feature count: 75 | max_evals: 151
✅ BG200 n = 200 | Train n = 463936
✅ Forced local samples from Hard. unique = 400 /400

=== Local SHAP (Soft) ===


PermutationExplainer explainer: 101it [01:56,  1.27s/it]                                                               


  chunk 0:100 done


PermutationExplainer explainer: 101it [01:55,  1.27s/it]                                                               


  chunk 100:200 done


PermutationExplainer explainer: 101it [01:57,  1.30s/it]                                                               


  chunk 200:300 done


PermutationExplainer explainer: 101it [01:59,  1.32s/it]                                                               


  chunk 300:400 done
✅ Local SHAP saved: .\outputs\ENSEMBLES\EnsembleSoft\local_xai_400\local_shap\shap_local_long.csv | rows: 30000

=== LIME (Soft) ===
  explained 50/400
  explained 100/400
  explained 150/400
  explained 200/400
  explained 250/400
  explained 300/400
  explained 350/400
  explained 400/400
✅ LIME saved: .\outputs\ENSEMBLES\EnsembleSoft\local_xai_400\lime\lime_explanations_long.csv | rows: 4000

✅ Intersection Hard vs Soft local samples: 400 /400


In [84]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================
OUT_FINAL = Path(r"outputs/_shared/final_package")
OUT_FINAL.mkdir(parents=True, exist_ok=True)

ENSEMBLES = {
    "EnsembleSoft": {
        "shap": r"outputs/ENSEMBLES/EnsembleSoft/local_xai_400/local_shap/shap_local_long.csv",
        "lime": r"outputs/ENSEMBLES/EnsembleSoft/local_xai_400/lime/lime_explanations_long.csv",
        "samples": r"outputs/ENSEMBLES/EnsembleSoft/local_xai_400/lime_samples_400.csv",
    },
    "EnsembleHard": {
        "shap": r"outputs/ENSEMBLES/EnsembleHard/local_xai_400/local_shap/shap_local_long.csv",
        "lime": r"outputs/ENSEMBLES/EnsembleHard/local_xai_400/lime/lime_explanations_long.csv",
        "samples": r"outputs/ENSEMBLES/EnsembleHard/local_xai_400/lime_samples_400.csv",
    },
    "EnsembleStacking": {
        "shap": r"outputs/ENSEMBLES/EnsembleStacking/local_xai_400/local_shap/shap_local_long.csv",
        "lime": r"outputs/ENSEMBLES/EnsembleStacking/local_xai_400/lime/lime_explanations_long.csv",
        "samples": r"outputs/ENSEMBLES/EnsembleStacking/local_xai_400/lime_samples_400.csv",
    }
}

TOPK = 10
MASTER_SAMPLES_PATH = ENSEMBLES["EnsembleHard"]["samples"]  # master 400 = Hard
MASTER_ID_COL = "row_idx"


# =========================================================
# Helpers (robust)
# =========================================================
def detect_id_col(df):
    cands = ["row_idx", "sample_id", "test_index", "test_idx", "idx", "index", "id"]
    for c in cands:
        if c in df.columns:
            return c
    # fallback: first integer-like column
    for c in df.columns[:10]:
        if pd.api.types.is_numeric_dtype(df[c]):
            return c
    return df.columns[0]

def ensure_int_series(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def load_master_ids():
    m = pd.read_csv(MASTER_SAMPLES_PATH)
    if MASTER_ID_COL not in m.columns:
        mcol = detect_id_col(m)
        m = m.rename(columns={mcol: MASTER_ID_COL})
    ids = ensure_int_series(m[MASTER_ID_COL]).dropna().astype(int).tolist()
    return set(ids)

MASTER_IDS = load_master_ids()
assert len(MASTER_IDS) == 400, f"MASTER 400 bekleniyordu, geldi: {len(MASTER_IDS)}"


# -----------------------------
# Local SHAP frequency
# Supports:
# - rank_abs or rank exists (already computed)
# - else use abs(shap_value) per row_idx
# Columns:
#   id: row_idx/sample_id...
#   feature: feature or feature_name
#   shap: shap_value
# -----------------------------
def localshap_frequency(shap_path, ensemble_name, master_ids, topk=10):
    df = pd.read_csv(shap_path)

    id_col = detect_id_col(df)
    df[id_col] = ensure_int_series(df[id_col])

    # Some files may store Soft sample_id; but we already standardized outputs to row_idx.
    # Filter to master set (strict same-400)
    df = df[df[id_col].isin(list(master_ids))].copy()
    n_unique = df[id_col].nunique()

    # feature col
    feat_col = "feature_name" if "feature_name" in df.columns else ("feature" if "feature" in df.columns else None)
    if feat_col is None:
        raise ValueError(f"{ensemble_name} SHAP: feature kolonu yok. Kolonlar: {df.columns.tolist()}")

    # rank col
    rank_col = None
    if "rank_abs" in df.columns:
        rank_col = "rank_abs"
    elif "rank" in df.columns:
        rank_col = "rank"

    if rank_col is not None:
        df = df.sort_values([id_col, rank_col], ascending=[True, True])
        top = df.groupby(id_col).head(topk)
    else:
        if "shap_value" not in df.columns:
            raise ValueError(f"{ensemble_name} SHAP: shap_value yok. Kolonlar: {df.columns.tolist()}")
        df["_abs"] = np.abs(pd.to_numeric(df["shap_value"], errors="coerce"))
        df = df.sort_values([id_col, "_abs"], ascending=[True, False])
        top = df.groupby(id_col).head(topk)

    freq = top[feat_col].astype(str).str.strip().value_counts().reset_index()
    freq.columns = ["item", "count"]
    freq["ensemble"] = ensemble_name
    freq["method"] = "LocalSHAP"
    return freq[["ensemble","method","item","count"]], n_unique


# -----------------------------
# LIME frequency
# Expects:
#  - rule col: feature_rule preferred; else tries to find a likely text column
#  - rank exists OR weight/abs_weight exists
# -----------------------------
def lime_frequency(lime_path, ensemble_name, master_ids, topk=10):
    df = pd.read_csv(lime_path)

    id_col = detect_id_col(df)
    df[id_col] = ensure_int_series(df[id_col])

    df = df[df[id_col].isin(list(master_ids))].copy()
    n_unique = df[id_col].nunique()

    # rule column
    rule_col = None
    if "feature_rule" in df.columns:
        rule_col = "feature_rule"
    else:
        for c in df.columns:
            if c.lower() in ["rule","rule_text","item","explanation","condition","term","feature"]:
                rule_col = c
                break
    if rule_col is None:
        raise ValueError(f"{ensemble_name} LIME: rule kolonu bulunamadı. Kolonlar: {df.columns.tolist()}")

    # topk selection
    if "rank" in df.columns:
        df = df.sort_values([id_col, "rank"], ascending=[True, True])
        top = df.groupby(id_col).head(topk)
    else:
        wcol = "abs_weight" if "abs_weight" in df.columns else ("weight" if "weight" in df.columns else None)
        if wcol is None:
            raise ValueError(f"{ensemble_name} LIME: rank/weight yok. Kolonlar: {df.columns.tolist()}")
        df["_absw"] = np.abs(pd.to_numeric(df[wcol], errors="coerce"))
        df = df.sort_values([id_col, "_absw"], ascending=[True, False])
        top = df.groupby(id_col).head(topk)

    freq = top[rule_col].astype(str).str.strip().value_counts().reset_index()
    freq.columns = ["item", "count"]
    freq["ensemble"] = ensemble_name
    freq["method"] = "LIME"
    return freq[["ensemble","method","item","count"]], n_unique


# -----------------------------
# Convert a LIME rule text into a "feature name" guess
# Example: "hostname_length <= 13.00" -> hostname_length
# Example: "0.00 < has_www <= 1.00" -> has_www
# We’ll do a robust extraction:
#   - remove numbers/operators
#   - pick first token that looks like a column name (letters/underscore/dot)
# -----------------------------
_feat_pat = re.compile(r"[A-Za-z_][A-Za-z0-9_\.]*")

def lime_rule_to_feature(rule: str):
    if not isinstance(rule, str):
        return None
    # common pattern like "0.00 < has_www <= 1.00"
    tokens = _feat_pat.findall(rule)
    if not tokens:
        return None
    # sometimes "num_at.1" exists, keep dot
    # pick the token that is not 'and', etc.
    for t in tokens:
        if t.lower() not in {"and","or","not"}:
            return t
    return tokens[0]


# =========================================================
# 1) Compute frequencies for all 3 ensembles
# =========================================================
records = []
report = []

for ens, paths in ENSEMBLES.items():
    # LocalSHAP
    shap_freq, n_shap = localshap_frequency(paths["shap"], ens, MASTER_IDS, TOPK)
    records.append(shap_freq)
    report.append((ens, "LocalSHAP", n_shap))

    # LIME
    lime_freq, n_lime = lime_frequency(paths["lime"], ens, MASTER_IDS, TOPK)
    records.append(lime_freq)
    report.append((ens, "LIME", n_lime))

summary_long = pd.concat(records, ignore_index=True)
summary_long = summary_long.sort_values(["ensemble","method","count"], ascending=[True, True, False])

# Save long summary
summary_path = OUT_FINAL / "local_xai_feature_frequency_summary_all3.csv"
summary_long.to_csv(summary_path, index=False)

print("✅ Summary long saved:", summary_path)
print("✅ Unique sample coverage (should be 400 each):")
for r in report:
    print(" -", r)


# =========================================================
# 2) TABLO 1: LocalSHAP pivot (features)
# =========================================================
tablo1 = summary_long[summary_long["method"]=="LocalSHAP"].pivot_table(
    index="item", columns="ensemble", values="count", aggfunc="sum", fill_value=0
)

tablo1 = tablo1.sort_values(by=list(tablo1.columns), ascending=False)
tablo1_path = OUT_FINAL / "TABLO1_localshap_top10_feature_frequency_pivot.csv"
tablo1.to_csv(tablo1_path)

print("✅ TABLO1 saved:", tablo1_path)


# =========================================================
# 3) TABLO 2: LIME pivot (rules)
# =========================================================
tablo2 = summary_long[summary_long["method"]=="LIME"].pivot_table(
    index="item", columns="ensemble", values="count", aggfunc="sum", fill_value=0
)
tablo2 = tablo2.sort_values(by=list(tablo2.columns), ascending=False)
tablo2_path = OUT_FINAL / "TABLO2_lime_top10_rule_frequency_pivot.csv"
tablo2.to_csv(tablo2_path)

print("✅ TABLO2 saved:", tablo2_path)


# =========================================================
# 4) TABLO 3: Core features (set logic)
#    - Three-way intersection
#    - Only stacking
#    - Only hard
#    - Only soft
# Based on LocalSHAP features (not LIME rules).
# We define "appears" as count>0 in freq list.
# =========================================================
soft_set = set(tablo1.index[tablo1.get("EnsembleSoft", 0) > 0])
hard_set = set(tablo1.index[tablo1.get("EnsembleHard", 0) > 0])
stack_set = set(tablo1.index[tablo1.get("EnsembleStacking", 0) > 0])

core_all3 = sorted(list(soft_set & hard_set & stack_set))
only_soft = sorted(list(soft_set - (hard_set | stack_set)))
only_hard = sorted(list(hard_set - (soft_set | stack_set)))
only_stack = sorted(list(stack_set - (soft_set | hard_set)))

tablo3 = pd.DataFrame({
    "core_all3": pd.Series(core_all3, dtype="object"),
    "only_soft": pd.Series(only_soft, dtype="object"),
    "only_hard": pd.Series(only_hard, dtype="object"),
    "only_stacking": pd.Series(only_stack, dtype="object"),
})

tablo3_path = OUT_FINAL / "TABLO3_core_feature_sets_localshap.csv"
tablo3.to_csv(tablo3_path, index=False)
print("✅ TABLO3 saved:", tablo3_path)


# =========================================================
# 5) Extra: SHAP vs LIME agreement (feature-level)
#    - Map LIME rules -> features
#    - Build pivot by feature (LIME-derived)
# =========================================================
lime_long = summary_long[summary_long["method"]=="LIME"].copy()
lime_long["feature_guess"] = lime_long["item"].apply(lime_rule_to_feature)

lime_feat_pivot = lime_long.dropna(subset=["feature_guess"]).pivot_table(
    index="feature_guess", columns="ensemble", values="count", aggfunc="sum", fill_value=0
).sort_values(by=list(ENSEMBLES.keys()), ascending=False)

lime_feat_pivot_path = OUT_FINAL / "EXTRA_lime_rules_mapped_to_feature_frequency_pivot.csv"
lime_feat_pivot.to_csv(lime_feat_pivot_path)
print("✅ EXTRA (LIME→feature) saved:", lime_feat_pivot_path)

print("\n✅ DONE. Files written to:", OUT_FINAL)

✅ Summary long saved: outputs\_shared\final_package\local_xai_feature_frequency_summary_all3.csv
✅ Unique sample coverage (should be 400 each):
 - ('EnsembleSoft', 'LocalSHAP', 400)
 - ('EnsembleSoft', 'LIME', 400)
 - ('EnsembleHard', 'LocalSHAP', 400)
 - ('EnsembleHard', 'LIME', 400)
 - ('EnsembleStacking', 'LocalSHAP', 400)
 - ('EnsembleStacking', 'LIME', 400)
✅ TABLO1 saved: outputs\_shared\final_package\TABLO1_localshap_top10_feature_frequency_pivot.csv
✅ TABLO2 saved: outputs\_shared\final_package\TABLO2_lime_top10_rule_frequency_pivot.csv
✅ TABLO3 saved: outputs\_shared\final_package\TABLO3_core_feature_sets_localshap.csv
✅ EXTRA (LIME→feature) saved: outputs\_shared\final_package\EXTRA_lime_rules_mapped_to_feature_frequency_pivot.csv

✅ DONE. Files written to: outputs\_shared\final_package


In [85]:
import os

base_path = "outputs"   # outputs klasörünün adı buysa
keywords = ["ensemble", "shap", "lime"]

for root, dirs, files in os.walk(base_path):
    for file in files:
        if any(k in file.lower() for k in keywords):
            print(os.path.join(root, file))

outputs\CatBoost\cache_v1\shap_values.npz
outputs\CatBoost\global\shap_bar_top20.png
outputs\CatBoost\global\shap_importance.csv
outputs\CatBoost\global\shap_meta.csv
outputs\CatBoost\global\shap_summary.png
outputs\CatBoost\global\.ipynb_checkpoints\shap_summary-checkpoint.png
outputs\CatBoost\global_fulltest_approx\shap_bar_top20.png
outputs\CatBoost\global_fulltest_approx\shap_importance.csv
outputs\CatBoost\global_fulltest_approx\shap_meta.csv
outputs\CatBoost\global_fulltest_approx\shap_summary.png
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_importance_N1000.csv
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_importance_N1500.csv
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_importance_N200.csv
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_importance_N500.csv
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_importance_steps_merged.csv
outputs\CatBoost\global_shap_fixedstep_1500_200\shap_values_N1000.npz
outputs\CatBoost\global_shap_fixedstep_1500_20

In [1]:
# ============================================================
# ASAMA2 SETTINGS - FULL PERFORMANCE (NO KNN) + ENSEMBLES
# Models: LogReg, DecisionTree, RandomForest, XGBoost, LightGBM, CatBoost
# Ensembles: Hard, Soft, Stacking(meta=LogReg, cv=5)
# Full dataset -> train/test split -> evaluate on FULL test set
# Output: DataFrame + outputs/_shared/final_metrics/ensemble_performance_fulltest.csv
# ============================================================

import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier

# ---- HARD REQUIRE: these packages must exist (no skipping)
missing = []
try:
    from xgboost import XGBClassifier
except Exception:
    missing.append("xgboost")

try:
    from lightgbm import LGBMClassifier
except Exception:
    missing.append("lightgbm")

try:
    from catboost import CatBoostClassifier
except Exception:
    missing.append("catboost")

if missing:
    raise ImportError(
        f"❌ Eksik paket(ler): {missing}\n"
        "Bu hücre 'tüm modeller kesin olsun' dediğin için devam etmiyor.\n"
        "Kurulum:\n"
        "  pip install xgboost lightgbm catboost\n"
        "veya:\n"
        "  conda install -c conda-forge xgboost lightgbm catboost\n"
    )

# ----------------------------
# CONFIG (Asama2'den)
# ----------------------------
CONFIG = {
    "seed": 42,
    "test_size": 0.20,
    "label_col": "label",
    "url_col": "url",
    "output_dir": "outputs",
}

SEED = CONFIG["seed"]
np.random.seed(SEED)

# DATA PATH: Asama2'deki gibi (gerekirse düzenle)
DATA_PATH = os.path.expanduser("~/Bitirme_Projesi_Dataset_Olusturma/Makale/final_lexical_dns_whois_cleaned.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"❌ DATA_PATH bulunamadı:\n{DATA_PATH}\n"
        "Not: Eğer dosyan aynı klasördeyse DATA_PATH'i 'final_lexical_dns_whois_cleaned.csv' yap."
    )

df = pd.read_csv(DATA_PATH)
print("✅ Loaded:", df.shape)

label_col = CONFIG["label_col"]
url_col = CONFIG["url_col"] if CONFIG["url_col"] in df.columns else None

drop_cols = [label_col] + ([url_col] if url_col else [])
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[label_col].astype(int).copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=SEED,
    stratify=y
)

print("✅ Train:", X_train_raw.shape, " Test:", X_test_raw.shape)
print("✅ #Features:", len(feature_cols))

# ----------------------------
# preprocessors (Asama2'den)
# ----------------------------
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

LINEAR_DISTANCE = {"LogReg"}  # KNN çıkarıldı

# ----------------------------
# base models (Asama2'deki parametrelerle)
# ----------------------------
base_models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=SEED),
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=SEED, eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        random_state=SEED, n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=800, learning_rate=0.05, depth=8,
        random_seed=SEED, verbose=False
    )
}

# ----------------------------
# build + fit pipelines
# ----------------------------
pipelines = {}
for name, clf in base_models.items():
    pre = linear_preprocessor if name in LINEAR_DISTANCE else tree_preprocessor
    pipe = Pipeline([("preprocess", pre), ("model", clf)])
    print("🚀 Fitting:", name)
    pipe.fit(X_train_raw, y_train)
    pipelines[name] = pipe

print("✅ pipelines hazır:", list(pipelines.keys()))

# ----------------------------
# Ensemblers (Hard / Soft / Stacking) - Asama2 mantığı
# ----------------------------
estimators = [(name, pipe) for name, pipe in pipelines.items()]

ens_soft = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
ens_hard = VotingClassifier(estimators=estimators, voting="hard", n_jobs=-1)

ens_stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000, n_jobs=-1, random_state=SEED),
    stack_method="predict_proba",
    passthrough=False,
    cv=5,
    n_jobs=-1
)

print("🚀 Fitting: EnsembleSoft")
ens_soft.fit(X_train_raw, y_train)

print("🚀 Fitting: EnsembleHard")
ens_hard.fit(X_train_raw, y_train)

print("🚀 Fitting: EnsembleStacking")
ens_stacking.fit(X_train_raw, y_train)

# ----------------------------
# Evaluation helpers
# ----------------------------
def proba_1(model, X_):
    return model.predict_proba(X_)[:, 1]

def eval_one(name, model):
    y_pred = model.predict(X_test_raw)
    y_prob = proba_1(model, X_test_raw)
    return {
        "model": name,
        "n_train": len(X_train_raw),
        "n_test": len(X_test_raw),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    }

rows = []
for name, pipe in pipelines.items():
    rows.append(eval_one(name, pipe))

rows.append(eval_one("EnsembleSoft", ens_soft))
rows.append(eval_one("EnsembleHard", ens_hard))
rows.append(eval_one("EnsembleStacking", ens_stacking))

perf = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)

print("\n==============================")
print("✅ PERFORMANCE TABLE (sorted by F1)")
print("==============================")
display(perf)

# Save
SAVE_DIR = os.path.join(CONFIG["output_dir"], "_shared", "final_metrics")
os.makedirs(SAVE_DIR, exist_ok=True)

out_path = os.path.join(SAVE_DIR, "ensemble_performance_fulltest.csv")
perf.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n✅ Saved: {out_path}")

✅ Loaded: (579920, 77)
✅ Train: (463936, 75)  Test: (115984, 75)
✅ #Features: 75
🚀 Fitting: LogReg
🚀 Fitting: DecisionTree
🚀 Fitting: RandomForest
🚀 Fitting: XGBoost
🚀 Fitting: LightGBM
[LightGBM] [Info] Number of positive: 192677, number of negative: 271259
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.068188 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 463936, number of used features: 75
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.415309 -> initscore=-0.342059
[LightGBM] [Info] Start training from score -0.342059
🚀 Fitting: CatBoost
✅ pipelines hazır: ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
🚀 Fitting: EnsembleSoft
🚀 Fitting: EnsembleHard
🚀 Fitting: EnsembleStacking


AttributeError: This 'VotingClassifier' has no attribute 'predict_proba'

In [2]:
from sklearn.metrics import confusion_matrix, classification_report

def has_predict_proba(model):
    return hasattr(model, "predict_proba")

def safe_predict_proba_1(model, X_):
    """Return proba for class 1 if available, else None."""
    if has_predict_proba(model):
        try:
            p = model.predict_proba(X_)
            if p is not None and p.shape[1] >= 2:
                return p[:, 1]
        except Exception as e:
            print(f"   ⚠️ predict_proba failed: {e}")
    return None

def eval_one(name, model, verbose=True):
    if verbose:
        print("\n" + "="*70)
        print(f"🧪 Evaluating: {name}")
        print(f"   predict_proba available? -> {has_predict_proba(model)}")

    # predictions
    y_pred = model.predict(X_test_raw)

    # proba (only if available)
    y_prob = safe_predict_proba_1(model, X_test_raw)

    # core metrics (always)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1v = f1_score(y_test, y_pred, zero_division=0)

    # optional metrics (require probability)
    if y_prob is not None:
        roc = roc_auc_score(y_test, y_prob)
        pr  = average_precision_score(y_test, y_prob)
    else:
        roc = np.nan
        pr  = np.nan

    if verbose:
        # quick distribution + sanity
        uniq, cnt = np.unique(y_pred, return_counts=True)
        pred_dist = dict(zip(uniq.tolist(), cnt.tolist()))
        print(f"   ✅ Accuracy={acc:.4f} Precision={prec:.4f} Recall={rec:.4f} F1={f1v:.4f}")
        print(f"   📌 Pred label dist: {pred_dist}")

        cm = confusion_matrix(y_test, y_pred)
        print("   📌 Confusion Matrix [[TN FP],[FN TP]]:")
        print(cm)

        if y_prob is not None:
            print(f"   ✅ ROC-AUC={roc:.4f} PR-AUC={pr:.4f}")
            print(f"   📌 proba sample: min={y_prob.min():.4f} mean={y_prob.mean():.4f} max={y_prob.max():.4f}")
            print(f"   📌 first 5 probas: {np.round(y_prob[:5], 4)}")
        else:
            print("   ⛔ ROC-AUC / PR-AUC skipped (no predict_proba for this model).")

    return {
        "model": name,
        "n_train": len(X_train_raw),
        "n_test": len(X_test_raw),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1v,
        "roc_auc": roc,
        "pr_auc": pr,
        "has_predict_proba": has_predict_proba(model),
    }

rows = []

print("\n==============================")
print("🔍 Evaluating single models...")
print("==============================")
for name, pipe in pipelines.items():
    rows.append(eval_one(name, pipe, verbose=True))

print("\n==============================")
print("🔍 Evaluating ensembles...")
print("==============================")
rows.append(eval_one("EnsembleSoft", ens_soft, verbose=True))
rows.append(eval_one("EnsembleHard", ens_hard, verbose=True))      # ROC/PR will be NaN
rows.append(eval_one("EnsembleStacking", ens_stacking, verbose=True))

perf = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)

print("\n==============================")
print("✅ PERFORMANCE TABLE (sorted by F1)")
print("==============================")
display(perf)

# Save
SAVE_DIR = os.path.join(CONFIG["output_dir"], "_shared", "final_metrics")
os.makedirs(SAVE_DIR, exist_ok=True)

out_path = os.path.join(SAVE_DIR, "ensemble_performance_fulltest.csv")
perf.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n✅ Saved: {out_path}")


🔍 Evaluating single models...

🧪 Evaluating: LogReg
   predict_proba available? -> True
   ✅ Accuracy=0.9440 Precision=0.9439 Recall=0.9198 F1=0.9317
   📌 Pred label dist: {0: 69042, 1: 46942}
   📌 Confusion Matrix [[TN FP],[FN TP]]:
[[65180  2635]
 [ 3862 44307]]
   ✅ ROC-AUC=0.9845 PR-AUC=0.9804
   📌 proba sample: min=0.0000 mean=0.4153 max=1.0000
   📌 first 5 probas: [9.745e-01 9.000e-04 4.530e-02 1.870e-02 4.300e-03]

🧪 Evaluating: DecisionTree
   predict_proba available? -> True
   ✅ Accuracy=0.9749 Precision=0.9721 Recall=0.9673 F1=0.9697
   📌 Pred label dist: {0: 68054, 1: 47930}
   📌 Confusion Matrix [[TN FP],[FN TP]]:
[[66478  1337]
 [ 1576 46593]]
   ✅ ROC-AUC=0.9738 PR-AUC=0.9539
   📌 proba sample: min=0.0000 mean=0.4132 max=1.0000
   📌 first 5 probas: [1. 0. 0. 0. 0.]

🧪 Evaluating: RandomForest
   predict_proba available? -> True
   ✅ Accuracy=0.9851 Precision=0.9905 Recall=0.9736 F1=0.9820
   📌 Pred label dist: {0: 68638, 1: 47346}
   📌 Confusion Matrix [[TN FP],[FN TP]]

,model,n_train,n_test,accuracy,precision,recall,f1,roc_auc,pr_auc,has_predict_proba
0,EnsembleStacking,463936,115984,0.988473,0.990572,0.981586,0.986059,0.998506,0.998209,True
1,LightGBM,463936,115984,0.986809,0.989833,0.978285,0.984025,0.998488,0.998241,True
2,RandomForest,463936,115984,0.985145,0.990496,0.973572,0.981961,0.998327,0.997946,True
3,EnsembleSoft,463936,115984,0.985119,0.988781,0.975233,0.981960,0.998106,0.997765,True
4,EnsembleHard,463936,115984,0.984472,0.991041,0.971392,0.981118,NaN,NaN,False
5,CatBoost,463936,115984,0.983153,0.986259,0.972991,0.979580,0.997907,0.997554,True
6,XGBoost,463936,115984,0.981592,0.984385,0.971081,0.977688,0.997635,0.997215,True
7,DecisionTree,463936,115984,0.974884,0.972105,0.967282,0.969688,0.973783,0.953888,True
8,LogReg,463936,115984,0.943984,0.943867,0.919824,0.931690,0.984539,0.980366,True



✅ Saved: outputs\_shared\final_metrics\ensemble_performance_fulltest.csv


In [ ]:
import time, json
import shap
import numpy as np
import pandas as pd
import os

# ----------------------------
# GLOBAL SHAP CONFIG
# ----------------------------
GLOBAL_SHAP_CFG = {
    "seed": SEED,
    "bg_size": 200,         # background sample (sen daha önce BG200 kullanıyordun)
    "eval_size": 1500,      # global shap hesaplanacak örnek sayısı (testten)
    "max_evals": 151,       # permutation explainer için tipik: 2*M + 1 (M=75 -> 151)
    "save_dir": os.path.join(CONFIG["output_dir"], "_shared", "global_shap_ensembles"),
    "use_hard_proxy": True, # Hard için proxy proba kullan (False -> hard shap skip)
}

os.makedirs(GLOBAL_SHAP_CFG["save_dir"], exist_ok=True)
rng = np.random.default_rng(GLOBAL_SHAP_CFG["seed"])

# ----------------------------
# helpers
# ----------------------------
def sample_rows(X_df, n, seed=42):
    n = min(n, len(X_df))
    return X_df.sample(n=n, random_state=seed).copy()

def ensure_float_df(X_df):
    # shap bazen object dtype sevmiyor; gerekirse düzelt
    return X_df.astype(np.float64, errors="ignore")

def predict_proba_wrapper(model, X_np_or_df):
    # SHAP, numpy array veya df gönderebilir; biz DataFrame'e çevirelim ki pipeline uyumlu kalsın
    if isinstance(X_np_or_df, np.ndarray):
        X_in = pd.DataFrame(X_np_or_df, columns=feature_cols)
    else:
        X_in = X_np_or_df
    return model.predict_proba(X_in)[:, 1]

def hard_proxy_proba(X_np_or_df):
    """
    Hard ensemble için proxy proba:
    base modellerin predict_proba ortalaması.
    (Hard'ın gerçek çıktısı değil; SHAP için süreklilik sağlar.)
    """
    if isinstance(X_np_or_df, np.ndarray):
        X_in = pd.DataFrame(X_np_or_df, columns=feature_cols)
    else:
        X_in = X_np_or_df

    probs = []
    for name, pipe in pipelines.items():
        if hasattr(pipe, "predict_proba"):
            probs.append(pipe.predict_proba(X_in)[:, 1])
        else:
            # teorik olarak burada olmamalı (senin tüm base modeller proba üretiyor)
            probs.append(pipe.predict(X_in).astype(float))

    return np.mean(np.vstack(probs), axis=0)

def save_global_shap_csvs(tag, shap_values, X_eval_df, out_dir):
    """
    shap_values: shape (n_samples, n_features)
    Kaydedilen CSV'ler:
      1) global_shap_importance.csv (feature bazlı)
      2) global_shap_values.csv (sample-feature matris)
      3) global_shap_values_summary.csv (sample bazlı özet: abs_sum, pred_prob, label, correct)
      4) meta.json (ayarlar)
    """
    os.makedirs(out_dir, exist_ok=True)

    sv = np.array(shap_values)
    if sv.ndim == 3:
        # bazı explainers class dimension döndürebilir; binary için genelde (n, m) bekleriz
        sv = sv[:, :, 0]

    # --- importance table
    mean_shap = sv.mean(axis=0)
    mean_abs = np.abs(sv).mean(axis=0)

    imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_shap": mean_shap,
        "mean_abs_shap": mean_abs,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    imp["rank_abs"] = np.arange(1, len(imp) + 1)

    imp_path = os.path.join(out_dir, "global_shap_importance.csv")
    imp.to_csv(imp_path, index=False, encoding="utf-8")

    # --- shap matrix (wide)
    shap_df = pd.DataFrame(sv, columns=feature_cols)
    shap_df.insert(0, "row_id", X_eval_df.index.values)
    shap_values_path = os.path.join(out_dir, "global_shap_values.csv")
    shap_df.to_csv(shap_values_path, index=False, encoding="utf-8")

    # --- per-sample summary
    # model proba + correctness ekleyelim (varsa)
    # Not: tag'e göre proba kaynağı farklı olabilir, aşağıda caller set edecek.
    summary = pd.DataFrame({
        "row_id": X_eval_df.index.values,
        "abs_shap_sum": np.abs(sv).sum(axis=1),
    })
    summary_path = os.path.join(out_dir, "global_shap_values_summary.csv")
    summary.to_csv(summary_path, index=False, encoding="utf-8")

    return {
        "importance_csv": imp_path,
        "values_csv": shap_values_path,
        "summary_csv": summary_path,
        "top10_features": imp["feature"].head(10).tolist()
    }

# ----------------------------
# prepare BG + EVAL sets
# ----------------------------
X_bg = sample_rows(X_train_raw, GLOBAL_SHAP_CFG["bg_size"], seed=GLOBAL_SHAP_CFG["seed"])
X_eval = sample_rows(X_test_raw, GLOBAL_SHAP_CFG["eval_size"], seed=GLOBAL_SHAP_CFG["seed"] + 7)

X_bg = ensure_float_df(X_bg)
X_eval = ensure_float_df(X_eval)

print(f"✅ BG size: {len(X_bg)} | Eval size: {len(X_eval)} | #features: {len(feature_cols)}")
print(f"✅ max_evals: {GLOBAL_SHAP_CFG['max_evals']}")

# ----------------------------
# run GLOBAL SHAP for ensembles
# ----------------------------
ensemble_targets = {
    "EnsembleSoft": ("proba", ens_soft),
    "EnsembleStacking": ("proba", ens_stacking),
    "EnsembleHard": ("hard_proxy" if GLOBAL_SHAP_CFG["use_hard_proxy"] else "skip", ens_hard),
}

results = []

for ens_name, (mode, model) in ensemble_targets.items():
    print("\n" + "="*80)
    print(f"🔎 GLOBAL SHAP 시작: {ens_name} | mode={mode}")

    if mode == "skip":
        print("⛔ Skipped (hard voting proba yok).")
        continue

    t0 = time.perf_counter()

    if mode == "proba":
        f = lambda X_: predict_proba_wrapper(model, X_)
        pred_prob = f(X_eval)
    elif mode == "hard_proxy":
        f = lambda X_: hard_proxy_proba(X_)
        pred_prob = f(X_eval)
    else:
        raise ValueError("Unknown mode")

    # Ara çıktı
    print(f"   📌 pred_prob: min={pred_prob.min():.4f} mean={pred_prob.mean():.4f} max={pred_prob.max():.4f}")

    # PermutationExplainer (model-agnostic, pipeline/ensemble ile uyumlu)
    explainer = shap.PermutationExplainer(f, X_bg, max_evals=GLOBAL_SHAP_CFG["max_evals"])

    print("   🚀 Explaining (this is the heavy part)...")
    exp = explainer(X_eval)

    shap_vals = exp.values  # (n_samples, n_features)

    out_dir = os.path.join(GLOBAL_SHAP_CFG["save_dir"], ens_name)
    saved = save_global_shap_csvs(ens_name, shap_vals, X_eval, out_dir)

    # summary csv'ye proba/label/correct ekleme (detay)
    y_eval = y_test.loc[X_eval.index] if isinstance(y_test, pd.Series) else pd.Series(y_test, index=X_test_raw.index).loc[X_eval.index]
    y_pred = (pred_prob >= 0.5).astype(int)
    correct = (y_pred == y_eval.values).astype(int)

    summary_path = saved["summary_csv"]
    summary_df = pd.read_csv(summary_path)
    summary_df["pred_prob"] = pred_prob
    summary_df["pred_label_05"] = y_pred
    summary_df["true_label"] = y_eval.values
    summary_df["correct_05"] = correct
    summary_df.to_csv(summary_path, index=False, encoding="utf-8")

    meta = {
        "ensemble": ens_name,
        "mode": mode,
        "bg_size": GLOBAL_SHAP_CFG["bg_size"],
        "eval_size": len(X_eval),
        "max_evals": GLOBAL_SHAP_CFG["max_evals"],
        "seed": GLOBAL_SHAP_CFG["seed"],
        "feature_count": len(feature_cols),
        "files": saved,
    }
    with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as fmeta:
        json.dump(meta, fmeta, ensure_ascii=False, indent=2)

    dt = time.perf_counter() - t0
    print(f"✅ Done: {ens_name} | saved to: {out_dir}")
    print(f"   🔥 Top10: {saved['top10_features']}")
    print(f"   ⏱️ Elapsed (seconds): {dt:.2f}")

    results.append({"ensemble": ens_name, "mode": mode, "elapsed_sec": dt, "out_dir": out_dir})

print("\n==============================")
print("✅ GLOBAL SHAP ensembles finished")
print("==============================")
display(pd.DataFrame(results))

✅ BG size: 200 | Eval size: 1500 | #features: 75
✅ max_evals: 151

🔎 GLOBAL SHAP 시작: EnsembleSoft | mode=proba
   📌 pred_prob: min=0.0000 mean=0.4132 max=1.0000
   🚀 Explaining (this is the heavy part)...


PermutationExplainer explainer: 1501it [27:19,  1.10s/it]                                                              


✅ Done: EnsembleSoft | saved to: outputs\_shared\global_shap_ensembles\EnsembleSoft
   🔥 Top10: ['hostname_length', 'has_www', 'domain_age_days', 'entropy_path', 'has_mx_record', 'uses_https', 'ttl_value', 'num_dots', 'domain_length', 'entropy_url']
   ⏱️ Elapsed (seconds): 1639.87

🔎 GLOBAL SHAP 시작: EnsembleStacking | mode=proba
   📌 pred_prob: min=0.0010 mean=0.4119 max=0.9996
   🚀 Explaining (this is the heavy part)...


PermutationExplainer explainer: 1501it [26:45,  1.08s/it]                                                              


✅ Done: EnsembleStacking | saved to: outputs\_shared\global_shap_ensembles\EnsembleStacking
   🔥 Top10: ['hostname_length', 'has_www', 'domain_age_days', 'entropy_path', 'has_mx_record', 'uses_https', 'entropy_hostname', 'num_dots', 'ttl_value', 'path_length']
   ⏱️ Elapsed (seconds): 1605.64

🔎 GLOBAL SHAP 시작: EnsembleHard | mode=hard_proxy
   📌 pred_prob: min=0.0000 mean=0.4133 max=1.0000
   🚀 Explaining (this is the heavy part)...


PermutationExplainer explainer:   5%|██▍                                             | 76/1500 [01:19<25:43,  1.08s/it]